# Feedback-Locked Theta Analysis

This notebook analyzes **feedback-related theta (4-8 Hz)** oscillations in baseline runs of the tACS bandit task.

## Overview

The goal is to extract theta power time-locked to feedback onset and examine:
1. Whether theta power increases following feedback presentation
2. Whether this differs between reward and no-reward outcomes
3. Individual differences in feedback-related theta (FRθ) magnitude

## Key Challenge: Timestamp Alignment

The behavioral task and EEG recording run on **separate computers** without hardware synchronization. Both systems log Unix timestamps, but:
- **EEG timestamps**: Unix time in milliseconds (column 12 of .easy file)
- **Behavioral timestamps**: Unix time in seconds (`trial_start_time` in CSV)

We align these post-hoc by:
1. Computing the offset between EEG recording start and first behavioral event
2. Converting all behavioral times to EEG-relative times
3. Validating alignment via visual inspection of event timing

**Note**: This approach assumes minimal clock drift between computers. For a 6-minute run, typical PC clock drift is <50ms, which is acceptable for theta-band analyses (period ~125-250ms).

## 0. Setup and Configuration

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='loadtxt: input contained no data')

# --- Path configuration ---
REPO_ROOT = Path('..').resolve()
EEG_DIR = REPO_ROOT / 'data' / 'nic' / 'raw'
BEHAV_DIR = REPO_ROOT / 'data' / 'bandit'
OUTPUT_DIR = REPO_ROOT / 'derivatives' / 'eeg' / 'feedback_theta'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Subject configuration ---
# Mirrors the main EEG notebook
SUBJECT_INFO = {
    '10886': {'counterbalance': 'B', 'earclip': False},
    '10998': {'counterbalance': 'B', 'earclip': False},
    '11773': {'counterbalance': 'B', 'earclip': False},
    '10656': {'counterbalance': 'A', 'earclip': False},
    '10951': {'counterbalance': 'A', 'earclip': False},
    '10418': {'counterbalance': 'B', 'earclip': False},
    '10636': {'counterbalance': 'B', 'earclip': True},
    '11318': {'counterbalance': 'B', 'earclip': True},
    '10369': {'counterbalance': 'B', 'earclip': True}
}

# --- EEG constants ---
FS = 500                    # Sampling rate (Hz)
EEG_CH_IDX = [4, 5, 6]      # EEG-only recording channels (F4, P4, P3)
EEG_CH_LABELS = ['F4', 'P4', 'P3']

# --- Task timing ---
FEEDBACK_DELAY = 2.5        # Seconds from response to feedback onset

# --- Epoch parameters ---
EPOCH_PRE = 1.0             # Seconds before feedback
EPOCH_POST = 1.5            # Seconds after feedback
BASELINE_WIN = (-0.5, -0.1) # Baseline window relative to feedback (seconds)

# --- Frequency band definitions ---
THETA_BAND = (4, 8)
ALPHA_BAND = (8, 13)

# --- Preprocessing parameters ---
HIGHPASS_FREQ = 0.5         # High-pass filter cutoff (Hz)
ARTIFACT_THRESH_UV = 150    # Artifact rejection threshold (microvolts)

print(f'Repository root: {REPO_ROOT}')
print(f'EEG directory: {EEG_DIR}')
print(f'Behavioral directory: {BEHAV_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Subjects: {list(SUBJECT_INFO.keys())}')

## 1. Data Loading Utilities

In [ ]:
def load_eeg_run(easy_path):
    """
    Load EEG data from a NIC2 .easy file.
    
    Returns:
        dict with keys:
            'eeg': ndarray (samples x 3), microvolts (F4, P4, P3)
            'timestamps_unix_ms': ndarray, Unix timestamps in milliseconds
            'time_sec': ndarray, time in seconds relative to recording start
            'fs': sampling rate (Hz)
    """
    raw = np.loadtxt(easy_path)
    
    # Extract EEG channels and convert nV to µV
    eeg = raw[:, EEG_CH_IDX] / 1000.0
    
    # Extract Unix timestamps (column 12, milliseconds)
    timestamps_unix_ms = raw[:, 12]
    
    # Compute relative time in seconds
    time_sec = (timestamps_unix_ms - timestamps_unix_ms[0]) / 1000.0
    
    return {
        'eeg': eeg,
        'timestamps_unix_ms': timestamps_unix_ms,
        'time_sec': time_sec,
        'fs': FS,
        'duration_sec': time_sec[-1]
    }


def load_behavioral_run(csv_path):
    """
    Load behavioral data from a trial-level CSV.
    
    IMPORTANT: The task code records timestamps at the END of each trial
    (after feedback and ITI), not at trial start. We need to work backwards
    to estimate feedback onset time.
    
    Timing flow in task:
        fixation → slots → response → highlight → wait → feedback → ITI → save
    
    The 'timestamp' column = seconds since experiment start, recorded after ITI.
    
    To get feedback onset, we need to subtract:
        - ITI duration (iti column, in ms)
        - Outcome/feedback duration (fixed at 1.0s in config)
    
    Returns:
        DataFrame with computed feedback times (relative to run start)
    """
    df = pd.read_csv(csv_path)
    
    # Remove missed trials (NaN rt) - these have no feedback
    df = df.dropna(subset=['rt']).copy()
    
    # Convert RT from milliseconds to seconds
    df['rt_sec'] = df['rt'] / 1000.0
    
    # Convert ITI from milliseconds to seconds
    df['iti_sec'] = df['iti'] / 1000.0
    
    # The 'timestamp' column is recorded at trial end (after feedback + ITI)
    # Feedback duration is fixed at 1.0 seconds in the task config
    FEEDBACK_DURATION = 1.0  # seconds
    
    # Work backwards to get feedback onset time
    # timestamp = feedback_onset + FEEDBACK_DURATION + ITI
    # feedback_onset = timestamp - FEEDBACK_DURATION - ITI
    df['feedback_time'] = df['timestamp'] - FEEDBACK_DURATION - df['iti_sec']
    
    # Zero to first trial's feedback for relative timing
    first_feedback = df['feedback_time'].iloc[0]
    df['feedback_time_relative'] = df['feedback_time'] - first_feedback
    
    # Store the first feedback time for later alignment
    df.attrs['first_feedback_in_run'] = first_feedback
    
    return df


def get_latest_run_file(subject_dir, run_num):
    """
    Find the latest behavioral CSV for a given run (handles restarts).
    """
    pattern = str(subject_dir / f'*run-{run_num:02d}*.csv')
    files = glob.glob(pattern)
    
    if not files:
        return None
    
    # Return most recent by modification time
    return max(files, key=os.path.getmtime)


def find_eeg_run(eeg_dir, subject_id, run_num):
    """
    Find the EEG .easy file for a given subject and run.
    Handles duplicate files by preferring the longest recording.
    """
    pattern = str(eeg_dir / f'*sub-{subject_id}_Run*{run_num}*.easy')
    files = glob.glob(pattern)
    
    # Filter to exact run number match
    matches = []
    for f in files:
        fname = os.path.basename(f)
        # Extract run number from filename like "...sub-11318_Run 1.easy"
        try:
            run_str = fname.split('Run')[1].strip().split('.')[0].strip()
            if int(run_str) == run_num:
                matches.append(f)
        except (IndexError, ValueError):
            continue
    
    if not matches:
        return None
    
    # Return largest file (longest recording)
    return max(matches, key=os.path.getsize)


print('Data loading utilities defined.')

## 2. Preprocessing Functions

In [ ]:
def highpass_filter(data, cutoff=HIGHPASS_FREQ, fs=FS, order=4):
    """
    Apply zero-phase Butterworth high-pass filter.
    
    Parameters:
        data: ndarray (samples,) or (samples, channels)
        cutoff: cutoff frequency in Hz
        fs: sampling rate
        order: filter order
    
    Returns:
        Filtered data with same shape as input
    """
    nyq = fs / 2
    b, a = signal.butter(order, cutoff / nyq, btype='high')
    
    if data.ndim == 1:
        return signal.filtfilt(b, a, data)
    else:
        return np.column_stack([signal.filtfilt(b, a, data[:, i]) 
                                for i in range(data.shape[1])])


def bandpass_filter(data, band, fs=FS, order=4):
    """
    Apply zero-phase Butterworth bandpass filter.
    """
    nyq = fs / 2
    low, high = band
    b, a = signal.butter(order, [low / nyq, high / nyq], btype='band')
    
    if data.ndim == 1:
        return signal.filtfilt(b, a, data)
    else:
        return np.column_stack([signal.filtfilt(b, a, data[:, i]) 
                                for i in range(data.shape[1])])


def avg_rereference(eeg):
    """
    Average re-reference across channels.
    Used for subjects without earclip reference.
    """
    avg = np.mean(eeg, axis=1, keepdims=True)
    return eeg - avg


def detect_artifacts(eeg, threshold_uv=ARTIFACT_THRESH_UV):
    """
    Detect samples exceeding artifact threshold.
    
    Returns:
        Boolean mask (True = artifact)
    """
    return np.any(np.abs(eeg) > threshold_uv, axis=1)


def preprocess_eeg(eeg_data, has_earclip=False):
    """
    Full preprocessing pipeline for EEG data.
    
    Steps:
        1. High-pass filter at 0.5 Hz (removes slow drifts)
        2. Average re-reference (if no earclip)
        3. Detect artifacts
    
    Parameters:
        eeg_data: dict from load_eeg_run()
        has_earclip: whether subject has earclip reference
    
    Returns:
        dict with:
            'eeg': preprocessed EEG (samples x channels)
            'artifact_mask': boolean mask for artifact samples
            + original keys
    """
    eeg = eeg_data['eeg'].copy()
    
    # Step 1: High-pass filter
    eeg = highpass_filter(eeg, cutoff=HIGHPASS_FREQ)
    
    # Step 2: Re-reference (only if no earclip)
    if not has_earclip:
        eeg = avg_rereference(eeg)
    
    # Step 3: Detect artifacts
    artifact_mask = detect_artifacts(eeg)
    
    # Return updated dict
    result = eeg_data.copy()
    result['eeg'] = eeg
    result['artifact_mask'] = artifact_mask
    result['artifact_pct'] = 100 * np.mean(artifact_mask)
    
    return result


print('Preprocessing functions defined.')

## 3. Timestamp Alignment

In [ ]:
def align_timestamps(eeg_data, behav_data):
    """
    Align behavioral feedback times to EEG recording time.
    
    The behavioral task and EEG recording run on separate computers without
    hardware synchronization. We use a heuristic approach:
    
    1. Behavioral data has 'feedback_time_relative' (seconds from first feedback)
    2. We estimate when the first feedback occurred in the EEG recording
    3. Add this offset to get EEG-relative times
    
    Heuristic: The task typically starts 10-30 seconds into the EEG recording
    (time for experimenter to start EEG, then start behavioral task, then
    participant sees instructions and completes first trial).
    
    We estimate the offset such that the behavioral data fits within the
    EEG recording with reasonable margins.
    
    Parameters:
        eeg_data: dict from load_eeg_run()
        behav_data: DataFrame from load_behavioral_run() with 'feedback_time_relative'
    
    Returns:
        DataFrame with 'feedback_time_eeg' (EEG-relative seconds)
    """
    behav_data = behav_data.copy()
    
    eeg_duration = eeg_data['duration_sec']
    
    # Get the span of behavioral data
    behav_span = behav_data['feedback_time_relative'].max()  # Already zeroed to first feedback
    
    # Estimate when first feedback occurred in EEG recording
    # Typical timing: EEG starts → ~10-30s setup → task starts → first trial ~5-10s
    # So first feedback is roughly 15-40 seconds into EEG recording
    
    # We want the data centered in the recording if possible
    # Or at minimum, with enough margin for epoching
    
    # Margin must accommodate the epoch window
    margin_start = EPOCH_PRE + 0.5   # Need EPOCH_PRE before first event, plus buffer
    margin_end = EPOCH_POST + 0.5    # Need EPOCH_POST after last event, plus buffer
    
    # Check if task fits in recording
    if behav_span > eeg_duration:
        # Task genuinely too long - this shouldn't happen
        raise ValueError(f'Behavioral data ({behav_span:.1f}s) exceeds EEG duration ({eeg_duration:.1f}s)')
    
    # Default: first feedback at ~20s into recording
    # Adjust if necessary to fit
    estimated_first_feedback = 20.0
    
    # Make sure last feedback fits (with margin for epoch window)
    last_feedback_time = estimated_first_feedback + behav_span
    if last_feedback_time > eeg_duration - margin_end:
        # Shift earlier
        estimated_first_feedback = eeg_duration - margin_end - behav_span
    
    # Make sure first feedback has margin (but allow tight fits)
    if estimated_first_feedback < margin_start:
        # Very tight fit - center the data as best we can
        available_slack = eeg_duration - behav_span - margin_start - margin_end
        if available_slack < 0:
            # Can't fit with full margins - use minimal margins
            estimated_first_feedback = EPOCH_PRE + 0.1
        else:
            estimated_first_feedback = margin_start
    
    # Compute EEG-relative feedback times
    behav_data['feedback_time_eeg'] = behav_data['feedback_time_relative'] + estimated_first_feedback
    
    # Store alignment metadata
    behav_data.attrs['estimated_first_feedback'] = estimated_first_feedback
    behav_data.attrs['offset_sec'] = estimated_first_feedback
    behav_data.attrs['behav_span'] = behav_span
    behav_data.attrs['alignment_method'] = 'heuristic'
    
    return behav_data


def validate_alignment(eeg_data, behav_data):
    """
    Check whether behavioral events fall within EEG recording window.
    
    Returns:
        dict with validation results
    """
    eeg_duration = eeg_data['duration_sec']
    feedback_times = behav_data['feedback_time_eeg'].values
    
    # Check bounds (with epoch margins)
    min_time = feedback_times.min() - EPOCH_PRE
    max_time = feedback_times.max() + EPOCH_POST
    
    within_bounds = (min_time >= 0) and (max_time <= eeg_duration)
    n_valid = np.sum((feedback_times - EPOCH_PRE >= 0) & 
                     (feedback_times + EPOCH_POST <= eeg_duration))
    
    return {
        'eeg_duration_sec': eeg_duration,
        'first_feedback_sec': feedback_times.min(),
        'last_feedback_sec': feedback_times.max(),
        'behav_span_sec': behav_data.attrs.get('behav_span', feedback_times.max() - feedback_times.min()),
        'offset_sec': behav_data.attrs.get('offset_sec', np.nan),
        'alignment_method': behav_data.attrs.get('alignment_method', 'unknown'),
        'within_bounds': within_bounds,
        'n_trials': len(feedback_times),
        'n_valid_epochs': n_valid
    }


print('Timestamp alignment functions defined.')

## 4. Epoching Functions

In [ ]:
def extract_epochs(eeg_data, event_times, pre_sec=EPOCH_PRE, post_sec=EPOCH_POST, 
                   reject_artifacts=True):
    """
    Extract epochs time-locked to events.
    
    Parameters:
        eeg_data: dict from preprocess_eeg() with 'eeg' and 'artifact_mask'
        event_times: array of event times in EEG-relative seconds
        pre_sec: seconds before event
        post_sec: seconds after event
        reject_artifacts: whether to reject epochs containing artifacts
    
    Returns:
        dict with:
            'epochs': ndarray (n_epochs x n_samples x n_channels)
            'times': ndarray, time axis relative to event (seconds)
            'valid_idx': indices of valid (non-rejected) epochs
            'reject_idx': indices of rejected epochs
    """
    eeg = eeg_data['eeg']
    fs = eeg_data['fs']
    artifact_mask = eeg_data.get('artifact_mask', np.zeros(len(eeg), dtype=bool))
    
    # Compute epoch sample indices
    n_pre = int(pre_sec * fs)
    n_post = int(post_sec * fs)
    n_samples = n_pre + n_post
    n_channels = eeg.shape[1]
    
    # Time axis
    times = np.linspace(-pre_sec, post_sec, n_samples, endpoint=False)
    
    # Extract epochs
    epochs = []
    valid_idx = []
    reject_idx = []
    
    for i, t in enumerate(event_times):
        # Convert to samples
        center_sample = int(t * fs)
        start_sample = center_sample - n_pre
        end_sample = center_sample + n_post
        
        # Check bounds
        if start_sample < 0 or end_sample > len(eeg):
            reject_idx.append(i)
            continue
        
        # Extract segment
        segment = eeg[start_sample:end_sample, :]
        
        # Check for artifacts
        if reject_artifacts and np.any(artifact_mask[start_sample:end_sample]):
            reject_idx.append(i)
            continue
        
        epochs.append(segment)
        valid_idx.append(i)
    
    if len(epochs) == 0:
        return {
            'epochs': np.zeros((0, n_samples, n_channels)),
            'times': times,
            'valid_idx': np.array([], dtype=int),
            'reject_idx': np.array(reject_idx, dtype=int),
            'n_valid': 0,
            'n_rejected': len(reject_idx)
        }
    
    return {
        'epochs': np.stack(epochs, axis=0),
        'times': times,
        'valid_idx': np.array(valid_idx, dtype=int),
        'reject_idx': np.array(reject_idx, dtype=int),
        'n_valid': len(epochs),
        'n_rejected': len(reject_idx)
    }


def baseline_correct(epochs, times, baseline_win=BASELINE_WIN):
    """
    Subtract baseline mean from each epoch.
    
    Parameters:
        epochs: ndarray (n_epochs x n_samples x n_channels)
        times: time axis (seconds)
        baseline_win: (start, end) in seconds
    
    Returns:
        Baseline-corrected epochs
    """
    baseline_mask = (times >= baseline_win[0]) & (times < baseline_win[1])
    baseline_mean = epochs[:, baseline_mask, :].mean(axis=1, keepdims=True)
    return epochs - baseline_mean


print('Epoching functions defined.')

## 5. Theta Power Analysis

In [ ]:
def compute_band_power_timecourse(epochs, times, band=THETA_BAND, fs=FS):
    """
    Compute instantaneous band power for each epoch.
    
    Method:
        1. Bandpass filter to isolate frequency band
        2. Compute Hilbert envelope (instantaneous amplitude)
        3. Square to get power
    
    Parameters:
        epochs: ndarray (n_epochs x n_samples x n_channels)
        times: time axis (seconds)
        band: (low, high) frequency range
        fs: sampling rate
    
    Returns:
        ndarray (n_epochs x n_samples x n_channels) of power values
    """
    n_epochs, n_samples, n_channels = epochs.shape
    power = np.zeros_like(epochs)
    
    for ch in range(n_channels):
        for ep in range(n_epochs):
            # Bandpass filter
            filtered = bandpass_filter(epochs[ep, :, ch], band, fs)
            
            # Hilbert envelope
            analytic = signal.hilbert(filtered)
            envelope = np.abs(analytic)
            
            # Power = amplitude squared
            power[ep, :, ch] = envelope ** 2
    
    return power


def compute_ersp(epochs, times, band=THETA_BAND, baseline_win=BASELINE_WIN, fs=FS):
    """
    Compute Event-Related Spectral Perturbation (ERSP).
    
    ERSP = (power - baseline_power) / baseline_power * 100
    
    This gives percent change from baseline, making it comparable
    across subjects and conditions with different absolute power levels.
    
    Returns:
        dict with:
            'ersp': ndarray (n_samples x n_channels), percent change
            'power': ndarray (n_epochs x n_samples x n_channels)
            'times': time axis
    """
    # Get raw power
    power = compute_band_power_timecourse(epochs, times, band, fs)
    
    # Compute baseline power per epoch per channel
    baseline_mask = (times >= baseline_win[0]) & (times < baseline_win[1])
    baseline_power = power[:, baseline_mask, :].mean(axis=1, keepdims=True)
    
    # Avoid division by zero
    baseline_power = np.maximum(baseline_power, 1e-10)
    
    # Compute ERSP per epoch
    ersp_epochs = (power - baseline_power) / baseline_power * 100
    
    # Average across epochs
    ersp = ersp_epochs.mean(axis=0)
    
    return {
        'ersp': ersp,
        'power': power,
        'times': times
    }


print('Theta power analysis functions defined.')

## 6. Single-Subject Analysis Pipeline

In [ ]:
def analyze_subject_run(subject_id, run_num, verbose=True):
    """
    Full analysis pipeline for a single subject and run.
    
    Steps:
        1. Load EEG and behavioral data
        2. Preprocess EEG
        3. Align timestamps
        4. Extract epochs around feedback
        5. Compute theta ERSP
        6. Split by reward/no-reward
    
    Returns:
        dict with all results, or None if data not found
    """
    if verbose:
        print(f'\n--- Sub-{subject_id}, Run {run_num} ---')
    
    # Get subject info
    sub_info = SUBJECT_INFO.get(subject_id, {})
    has_earclip = sub_info.get('earclip', False)
    
    # --- Load EEG ---
    eeg_path = find_eeg_run(EEG_DIR, subject_id, run_num)
    if eeg_path is None:
        if verbose:
            print(f'  EEG file not found')
        return None
    
    eeg_data = load_eeg_run(eeg_path)
    if verbose:
        print(f'  EEG loaded: {eeg_data["duration_sec"]:.1f}s')
    
    # --- Load behavioral ---
    behav_dir = BEHAV_DIR / f'sub-{subject_id}'
    behav_path = get_latest_run_file(behav_dir, run_num)
    if behav_path is None:
        if verbose:
            print(f'  Behavioral file not found')
        return None
    
    behav_data = load_behavioral_run(behav_path)
    if verbose:
        print(f'  Behavioral loaded: {len(behav_data)} trials')
    
    # --- Preprocess EEG ---
    eeg_data = preprocess_eeg(eeg_data, has_earclip=has_earclip)
    if verbose:
        print(f'  Artifacts: {eeg_data["artifact_pct"]:.1f}%')
    
    # --- Align timestamps ---
    behav_data = align_timestamps(eeg_data, behav_data)
    
    # --- Validate alignment ---
    alignment = validate_alignment(eeg_data, behav_data)
    if verbose:
        print(f'  Alignment method: {alignment["alignment_method"]}')
        print(f'  Behavioral span: {alignment["behav_span_sec"]:.1f}s, EEG duration: {alignment["eeg_duration_sec"]:.1f}s')
        print(f'  Estimated offset: {alignment["offset_sec"]:.1f}s (first feedback in EEG time)')
        print(f'  Feedback range: {alignment["first_feedback_sec"]:.1f}s - {alignment["last_feedback_sec"]:.1f}s')
        print(f'  Within bounds: {alignment["within_bounds"]}')
    
    if not alignment['within_bounds']:
        if verbose:
            print(f'  WARNING: Events outside EEG recording window!')
        return None
    
    # --- Extract epochs ---
    feedback_times = behav_data['feedback_time_eeg'].values
    epoch_data = extract_epochs(eeg_data, feedback_times)
    
    if epoch_data['n_valid'] < 10:
        if verbose:
            print(f'  Too few valid epochs: {epoch_data["n_valid"]}')
        return None
    
    if verbose:
        print(f'  Epochs: {epoch_data["n_valid"]} valid, {epoch_data["n_rejected"]} rejected')
    
    # --- Compute theta ERSP ---
    ersp_data = compute_ersp(epoch_data['epochs'], epoch_data['times'], THETA_BAND)
    
    # --- Split by reward outcome ---
    valid_trials = behav_data.iloc[epoch_data['valid_idx']].copy()
    
    reward_mask = valid_trials['reward'].values.astype(bool)
    n_reward = np.sum(reward_mask)
    n_no_reward = len(reward_mask) - n_reward
    
    if verbose:
        print(f'  Reward trials: {n_reward}, No-reward trials: {n_no_reward}')
    
    # Compute ERSP separately for reward and no-reward
    if n_reward >= 5:
        ersp_reward = compute_ersp(
            epoch_data['epochs'][reward_mask], 
            epoch_data['times'], 
            THETA_BAND
        )
    else:
        ersp_reward = None
    
    if n_no_reward >= 5:
        ersp_no_reward = compute_ersp(
            epoch_data['epochs'][~reward_mask], 
            epoch_data['times'], 
            THETA_BAND
        )
    else:
        ersp_no_reward = None
    
    return {
        'subject_id': subject_id,
        'run_num': run_num,
        'has_earclip': has_earclip,
        'alignment': alignment,
        'n_trials': len(behav_data),
        'n_valid_epochs': epoch_data['n_valid'],
        'n_rejected_epochs': epoch_data['n_rejected'],
        'n_reward': n_reward,
        'n_no_reward': n_no_reward,
        'times': ersp_data['times'],
        'ersp_all': ersp_data['ersp'],
        'ersp_reward': ersp_reward['ersp'] if ersp_reward else None,
        'ersp_no_reward': ersp_no_reward['ersp'] if ersp_no_reward else None,
    }


print('Single-subject pipeline defined.')

## 7. Test on Single Subject

In [ ]:
# Test on one subject (baseline run)
test_subject = '11318'
test_run = 1

result = analyze_subject_run(test_subject, test_run, verbose=True)

if result:
    print(f'\nAnalysis successful!')
else:
    print(f'\nAnalysis failed - check output above')

## 8. Visualize Results

In [ ]:
def plot_ersp(result, channel_idx=0):
    """
    Plot theta ERSP for a single subject/run.
    
    Parameters:
        result: dict from analyze_subject_run()
        channel_idx: which channel to plot (0=F4, 1=P4, 2=P3)
    """
    if result is None:
        print('No results to plot')
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    ch_label = EEG_CH_LABELS[channel_idx]
    times = result['times']
    
    # --- Plot 1: Overall ERSP ---
    ax1 = axes[0]
    ax1.plot(times, result['ersp_all'][:, channel_idx], 'k-', linewidth=2)
    ax1.axvline(0, color='r', linestyle='--', label='Feedback')
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax1.fill_between(times, 0, result['ersp_all'][:, channel_idx], 
                     where=result['ersp_all'][:, channel_idx] > 0, 
                     alpha=0.3, color='green')
    ax1.fill_between(times, 0, result['ersp_all'][:, channel_idx], 
                     where=result['ersp_all'][:, channel_idx] < 0, 
                     alpha=0.3, color='red')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Theta Power (% change)')
    ax1.set_title(f'Sub-{result["subject_id"]} Run {result["run_num"]}\n'
                  f'Feedback-locked Theta ({ch_label})')
    ax1.legend()
    ax1.set_xlim(times[0], times[-1])
    
    # --- Plot 2: Reward vs No-Reward ---
    ax2 = axes[1]
    if result['ersp_reward'] is not None:
        ax2.plot(times, result['ersp_reward'][:, channel_idx], 
                 'g-', linewidth=2, label=f'Reward (n={result["n_reward"]})')
    if result['ersp_no_reward'] is not None:
        ax2.plot(times, result['ersp_no_reward'][:, channel_idx], 
                 'r-', linewidth=2, label=f'No Reward (n={result["n_no_reward"]})')
    ax2.axvline(0, color='k', linestyle='--', alpha=0.5)
    ax2.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Theta Power (% change)')
    ax2.set_title(f'Reward vs No-Reward ({ch_label})')
    ax2.legend()
    ax2.set_xlim(times[0], times[-1])
    
    plt.tight_layout()
    plt.show()
    
    return fig


# Plot results
if result:
    fig = plot_ersp(result, channel_idx=0)  # F4

## 9. Run All Baseline Runs

In [ ]:
# Analyze baseline runs (1 and 5) for all subjects
baseline_runs = [1, 5]

all_results = []

for subject_id in SUBJECT_INFO.keys():
    for run_num in baseline_runs:
        result = analyze_subject_run(subject_id, run_num, verbose=True)
        
        if result:
            all_results.append(result)

print(f'\n=== Summary ===')
print(f'Successfully analyzed: {len(all_results)} runs')
print(f'Subjects with data: {len(set(r["subject_id"] for r in all_results))}')

## 10. Group Average

In [ ]:
def plot_group_average(results, channel_idx=0):
    """
    Plot group-average theta ERSP with SEM bands.
    """
    if len(results) == 0:
        print('No results to plot')
        return
    
    ch_label = EEG_CH_LABELS[channel_idx]
    times = results[0]['times']
    
    # Collect ERSPs
    ersp_all = np.stack([r['ersp_all'][:, channel_idx] for r in results])
    
    ersp_reward = [r['ersp_reward'][:, channel_idx] for r in results 
                   if r['ersp_reward'] is not None]
    ersp_no_reward = [r['ersp_no_reward'][:, channel_idx] for r in results 
                       if r['ersp_no_reward'] is not None]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # --- Plot 1: Overall ---
    ax1 = axes[0]
    mean_all = ersp_all.mean(axis=0)
    sem_all = ersp_all.std(axis=0) / np.sqrt(len(ersp_all))
    
    ax1.plot(times, mean_all, 'k-', linewidth=2)
    ax1.fill_between(times, mean_all - sem_all, mean_all + sem_all, 
                     alpha=0.3, color='gray')
    ax1.axvline(0, color='r', linestyle='--', label='Feedback')
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Theta Power (% change)')
    ax1.set_title(f'Group Average ({ch_label})\nn={len(ersp_all)} runs')
    ax1.legend()
    ax1.set_xlim(times[0], times[-1])
    
    # --- Plot 2: Reward vs No-Reward ---
    ax2 = axes[1]
    
    if len(ersp_reward) > 0:
        ersp_reward = np.stack(ersp_reward)
        mean_rew = ersp_reward.mean(axis=0)
        sem_rew = ersp_reward.std(axis=0) / np.sqrt(len(ersp_reward))
        ax2.plot(times, mean_rew, 'g-', linewidth=2, label=f'Reward (n={len(ersp_reward)})')
        ax2.fill_between(times, mean_rew - sem_rew, mean_rew + sem_rew, 
                         alpha=0.2, color='green')
    
    if len(ersp_no_reward) > 0:
        ersp_no_reward = np.stack(ersp_no_reward)
        mean_no = ersp_no_reward.mean(axis=0)
        sem_no = ersp_no_reward.std(axis=0) / np.sqrt(len(ersp_no_reward))
        ax2.plot(times, mean_no, 'r-', linewidth=2, label=f'No Reward (n={len(ersp_no_reward)})')
        ax2.fill_between(times, mean_no - sem_no, mean_no + sem_no, 
                         alpha=0.2, color='red')
    
    ax2.axvline(0, color='k', linestyle='--', alpha=0.5)
    ax2.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Theta Power (% change)')
    ax2.set_title(f'Reward vs No-Reward ({ch_label})')
    ax2.legend()
    ax2.set_xlim(times[0], times[-1])
    
    plt.tight_layout()
    plt.show()
    
    return fig


# Plot group average
if len(all_results) > 0:
    fig = plot_group_average(all_results, channel_idx=0)  # F4

## 11. Export Summary Statistics

In [ ]:
def compute_summary_stats(results):
    """
    Compute summary statistics for each subject/run.
    
    Extracts:
        - Mean theta power in post-feedback window (0 to 500ms)
        - Peak theta power and latency
        - Reward vs no-reward difference
    """
    rows = []
    
    for r in results:
        times = r['times']
        
        # Post-feedback window: 0 to 500ms
        post_mask = (times >= 0) & (times < 0.5)
        
        for ch_idx, ch_label in enumerate(EEG_CH_LABELS):
            ersp = r['ersp_all'][:, ch_idx]
            ersp_post = ersp[post_mask]
            
            row = {
                'subject_id': r['subject_id'],
                'run': r['run_num'],
                'channel': ch_label,
                'has_earclip': r['has_earclip'],
                'n_trials': r['n_trials'],
                'n_valid_epochs': r['n_valid_epochs'],
                'n_reward': r['n_reward'],
                'n_no_reward': r['n_no_reward'],
                'mean_theta_post': ersp_post.mean(),
                'peak_theta': ersp_post.max(),
                'peak_latency_ms': times[post_mask][np.argmax(ersp_post)] * 1000,
            }
            
            # Add reward/no-reward if available
            if r['ersp_reward'] is not None:
                row['mean_theta_reward'] = r['ersp_reward'][post_mask, ch_idx].mean()
            else:
                row['mean_theta_reward'] = np.nan
            
            if r['ersp_no_reward'] is not None:
                row['mean_theta_no_reward'] = r['ersp_no_reward'][post_mask, ch_idx].mean()
            else:
                row['mean_theta_no_reward'] = np.nan
            
            row['theta_diff'] = row['mean_theta_no_reward'] - row['mean_theta_reward']
            
            rows.append(row)
    
    return pd.DataFrame(rows)


if len(all_results) > 0:
    summary_df = compute_summary_stats(all_results)
    
    # Save to CSV
    output_path = OUTPUT_DIR / 'feedback_theta_summary.csv'
    summary_df.to_csv(output_path, index=False)
    print(f'Summary saved to: {output_path}')
    
    # Display
    display(summary_df)
else:
    print('No results to summarize')

## 12. Response-Locked Analysis (Disambiguation)

The feedback-locked analysis shows a prominent theta burst **before** feedback onset (around -0.9s). This could be:

1. **Response-locked theta**: Motor/decision-related theta that occurs around the button press
2. **Anticipatory theta**: Expectation-related activity during the wait period before feedback
3. **Timing misalignment**: Our heuristic offset might be wrong

To disambiguate, we epoch the same data around **response time** instead of feedback time.

### Trial timing structure:
```
Fixation → Stimulus → Response (RT) → Highlight (0.3s) → Wait (1.5-2.5s) → Feedback (1.0s) → ITI
                      ^                                                    ^
                      Response-locked t=0                                  Feedback-locked t=0
```

The response-to-feedback delay is: highlight (0.3s) + wait (~2.0s) + FEEDBACK_DELAY constant we used = **~2.5s total**

In [ ]:
def load_behavioral_with_response_times(csv_path):
    """
    Load behavioral data and compute BOTH response and feedback times.
    
    Returns DataFrame with:
        - response_time_relative: time of button press (zeroed to first response)
        - feedback_time_relative: time of feedback onset (zeroed to first feedback)
    """
    df = pd.read_csv(csv_path)
    
    # Remove missed trials
    df = df.dropna(subset=['rt']).copy()
    
    # Convert units
    df['rt_sec'] = df['rt'] / 1000.0
    df['iti_sec'] = df['iti'] / 1000.0
    df['wait_sec'] = df['wait_time'] / 1000.0
    
    # Task timing constants
    FEEDBACK_DURATION = 1.0  # seconds
    HIGHLIGHT_DURATION = 0.3  # seconds (choice highlight before wait)
    
    # The 'timestamp' column is recorded at trial END (after feedback + ITI)
    # Work backwards to get key event times:
    
    # Feedback onset = timestamp - feedback_duration - ITI
    df['feedback_time'] = df['timestamp'] - FEEDBACK_DURATION - df['iti_sec']
    
    # Response time = feedback - wait - highlight
    # (response happens, then highlight, then wait, then feedback)
    df['response_time'] = df['feedback_time'] - df['wait_sec'] - HIGHLIGHT_DURATION
    
    # Zero to first event for relative timing
    first_response = df['response_time'].iloc[0]
    first_feedback = df['feedback_time'].iloc[0]
    
    df['response_time_relative'] = df['response_time'] - first_response
    df['feedback_time_relative'] = df['feedback_time'] - first_feedback
    
    # Store metadata
    df.attrs['first_response_in_run'] = first_response
    df.attrs['first_feedback_in_run'] = first_feedback
    df.attrs['response_to_feedback_delay'] = first_feedback - first_response
    
    return df


def align_response_timestamps(eeg_data, behav_data):
    """
    Align response times to EEG recording.
    Must ensure BOTH response and feedback events fit within EEG window.
    """
    behav_data = behav_data.copy()
    
    eeg_duration = eeg_data['duration_sec']
    
    # Get span of response and feedback times
    response_span = behav_data['response_time_relative'].max()
    feedback_span = behav_data['feedback_time_relative'].max()
    resp_to_fb_delay = behav_data.attrs.get('response_to_feedback_delay', 2.5)
    
    # The constraint: we need room for:
    # - EPOCH_PRE before first response
    # - EPOCH_POST after last feedback
    # Last feedback = first_response + response_span + resp_to_fb_delay (approx)
    
    margin_start = EPOCH_PRE + 0.5
    margin_end = EPOCH_POST + 0.5
    
    # Total span from first response to last feedback
    total_span = feedback_span + resp_to_fb_delay
    
    # Default: first response at ~10s into recording
    estimated_first_response = 10.0
    
    # Make sure last feedback fits
    if estimated_first_response + total_span > eeg_duration - margin_end:
        estimated_first_response = eeg_duration - margin_end - total_span
    
    # Make sure first response has margin
    if estimated_first_response < margin_start:
        estimated_first_response = margin_start
    
    # Compute EEG-relative times
    behav_data['response_time_eeg'] = behav_data['response_time_relative'] + estimated_first_response
    behav_data['feedback_time_eeg'] = behav_data['feedback_time_relative'] + estimated_first_response + resp_to_fb_delay
    
    behav_data.attrs['estimated_first_response'] = estimated_first_response
    behav_data.attrs['response_span'] = response_span
    
    return behav_data


print('Response-locked functions defined.')

In [ ]:
def analyze_response_locked(subject_id, run_num, verbose=True):
    """
    Analyze BOTH response-locked and feedback-locked theta for comparison.
    
    Returns dict with both epoch types for direct comparison.
    """
    if verbose:
        print(f'\n--- Sub-{subject_id}, Run {run_num} (Response vs Feedback) ---')
    
    # Get subject info
    sub_info = SUBJECT_INFO.get(subject_id, {})
    has_earclip = sub_info.get('earclip', False)
    
    # --- Load EEG ---
    eeg_path = find_eeg_run(EEG_DIR, subject_id, run_num)
    if eeg_path is None:
        if verbose:
            print(f'  EEG file not found')
        return None
    
    eeg_data = load_eeg_run(eeg_path)
    
    # --- Load behavioral with both timestamps ---
    behav_dir = BEHAV_DIR / f'sub-{subject_id}'
    behav_path = get_latest_run_file(behav_dir, run_num)
    if behav_path is None:
        if verbose:
            print(f'  Behavioral file not found')
        return None
    
    behav_data = load_behavioral_with_response_times(behav_path)
    if verbose:
        delay = behav_data.attrs.get('response_to_feedback_delay', 0)
        print(f'  Loaded {len(behav_data)} trials, response-to-feedback delay: {delay:.2f}s')
    
    # --- Preprocess EEG ---
    eeg_data = preprocess_eeg(eeg_data, has_earclip=has_earclip)
    
    # --- Align timestamps ---
    behav_data = align_response_timestamps(eeg_data, behav_data)
    
    # Check bounds for both event types
    response_times = behav_data['response_time_eeg'].values
    feedback_times = behav_data['feedback_time_eeg'].values
    eeg_duration = eeg_data['duration_sec']
    
    # Validate
    resp_ok = (response_times.min() > EPOCH_PRE) and (response_times.max() < eeg_duration - EPOCH_POST)
    fb_ok = (feedback_times.min() > EPOCH_PRE) and (feedback_times.max() < eeg_duration - EPOCH_POST)
    
    if not (resp_ok and fb_ok):
        if verbose:
            print(f'  Events outside bounds')
        return None
    
    if verbose:
        print(f'  Response range: {response_times.min():.1f}s - {response_times.max():.1f}s')
        print(f'  Feedback range: {feedback_times.min():.1f}s - {feedback_times.max():.1f}s')
    
    # --- Extract RESPONSE-LOCKED epochs ---
    resp_epochs = extract_epochs(eeg_data, response_times)
    
    # --- Extract FEEDBACK-LOCKED epochs ---
    fb_epochs = extract_epochs(eeg_data, feedback_times)
    
    if resp_epochs['n_valid'] < 10 or fb_epochs['n_valid'] < 10:
        if verbose:
            print(f'  Too few epochs')
        return None
    
    if verbose:
        print(f'  Response epochs: {resp_epochs["n_valid"]} valid')
        print(f'  Feedback epochs: {fb_epochs["n_valid"]} valid')
    
    # --- Compute ERSP for both ---
    resp_ersp = compute_ersp(resp_epochs['epochs'], resp_epochs['times'], THETA_BAND)
    fb_ersp = compute_ersp(fb_epochs['epochs'], fb_epochs['times'], THETA_BAND)
    
    return {
        'subject_id': subject_id,
        'run_num': run_num,
        'times': resp_epochs['times'],  # Same for both
        'response_ersp': resp_ersp['ersp'],
        'feedback_ersp': fb_ersp['ersp'],
        'n_response_epochs': resp_epochs['n_valid'],
        'n_feedback_epochs': fb_epochs['n_valid'],
        'response_to_feedback_delay': behav_data.attrs.get('response_to_feedback_delay', 2.5),
    }


print('Response-locked analysis function defined.')

In [ ]:
# Test on single subject
test_result = analyze_response_locked('11318', 1, verbose=True)

if test_result:
    print('\nAnalysis successful!')

In [ ]:
def plot_response_vs_feedback(result, channel_idx=0):
    """
    Plot response-locked vs feedback-locked theta to disambiguate the source.
    """
    if result is None:
        print('No results to plot')
        return
    
    ch_label = EEG_CH_LABELS[channel_idx]
    times = result['times']
    delay = result['response_to_feedback_delay']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Left: Response-locked ---
    ax1 = axes[0]
    resp_ersp = result['response_ersp'][:, channel_idx]
    ax1.plot(times, resp_ersp, 'b-', linewidth=2)
    ax1.axvline(0, color='b', linestyle='--', linewidth=2, label='Response')
    ax1.axvline(delay, color='r', linestyle=':', linewidth=2, label=f'Feedback (+{delay:.1f}s)')
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax1.fill_between(times, 0, resp_ersp, where=resp_ersp > 0, alpha=0.3, color='blue')
    ax1.set_xlabel('Time from Response (s)')
    ax1.set_ylabel('Theta Power (% change)')
    ax1.set_title(f'Response-Locked Theta ({ch_label})\nn={result["n_response_epochs"]} epochs')
    ax1.legend(loc='upper right')
    ax1.set_xlim(times[0], times[-1])
    
    # --- Right: Feedback-locked ---
    ax2 = axes[1]
    fb_ersp = result['feedback_ersp'][:, channel_idx]
    ax2.plot(times, fb_ersp, 'r-', linewidth=2)
    ax2.axvline(0, color='r', linestyle='--', linewidth=2, label='Feedback')
    ax2.axvline(-delay, color='b', linestyle=':', linewidth=2, label=f'Response (-{delay:.1f}s)')
    ax2.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax2.fill_between(times, 0, fb_ersp, where=fb_ersp > 0, alpha=0.3, color='red')
    ax2.set_xlabel('Time from Feedback (s)')
    ax2.set_ylabel('Theta Power (% change)')
    ax2.set_title(f'Feedback-Locked Theta ({ch_label})\nn={result["n_feedback_epochs"]} epochs')
    ax2.legend(loc='upper right')
    ax2.set_xlim(times[0], times[-1])
    
    plt.suptitle(f'Sub-{result["subject_id"]} Run {result["run_num"]}: Response vs Feedback Locking', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return fig


# Plot single subject
if test_result:
    fig = plot_response_vs_feedback(test_result, channel_idx=0)

In [ ]:
# Run on all baseline runs
response_results = []

for subject_id in SUBJECT_INFO.keys():
    for run_num in [1, 5]:  # Baseline runs
        result = analyze_response_locked(subject_id, run_num, verbose=True)
        if result:
            response_results.append(result)

print(f'\n=== Summary ===')
print(f'Successfully analyzed: {len(response_results)} runs')

In [ ]:
def plot_group_response_vs_feedback(results, channel_idx=0):
    """
    Plot group-average response-locked vs feedback-locked theta.
    """
    if len(results) == 0:
        print('No results to plot')
        return
    
    ch_label = EEG_CH_LABELS[channel_idx]
    times = results[0]['times']
    
    # Collect ERSPs
    resp_ersp = np.stack([r['response_ersp'][:, channel_idx] for r in results])
    fb_ersp = np.stack([r['feedback_ersp'][:, channel_idx] for r in results])
    
    # Mean delay
    mean_delay = np.mean([r['response_to_feedback_delay'] for r in results])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Response-locked ---
    ax1 = axes[0]
    mean_resp = resp_ersp.mean(axis=0)
    sem_resp = resp_ersp.std(axis=0) / np.sqrt(len(resp_ersp))
    
    ax1.plot(times, mean_resp, 'b-', linewidth=2)
    ax1.fill_between(times, mean_resp - sem_resp, mean_resp + sem_resp, alpha=0.3, color='blue')
    ax1.axvline(0, color='b', linestyle='--', linewidth=2, label='Response')
    ax1.axvline(mean_delay, color='r', linestyle=':', linewidth=2, label=f'Feedback (+{mean_delay:.1f}s)')
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax1.set_xlabel('Time from Response (s)')
    ax1.set_ylabel('Theta Power (% change)')
    ax1.set_title(f'Response-Locked Theta ({ch_label})\nn={len(results)} runs')
    ax1.legend(loc='upper right')
    ax1.set_xlim(times[0], times[-1])
    
    # --- Feedback-locked ---
    ax2 = axes[1]
    mean_fb = fb_ersp.mean(axis=0)
    sem_fb = fb_ersp.std(axis=0) / np.sqrt(len(fb_ersp))
    
    ax2.plot(times, mean_fb, 'r-', linewidth=2)
    ax2.fill_between(times, mean_fb - sem_fb, mean_fb + sem_fb, alpha=0.3, color='red')
    ax2.axvline(0, color='r', linestyle='--', linewidth=2, label='Feedback')
    ax2.axvline(-mean_delay, color='b', linestyle=':', linewidth=2, label=f'Response (-{mean_delay:.1f}s)')
    ax2.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax2.set_xlabel('Time from Feedback (s)')
    ax2.set_ylabel('Theta Power (% change)')
    ax2.set_title(f'Feedback-Locked Theta ({ch_label})\nn={len(results)} runs')
    ax2.legend(loc='upper right')
    ax2.set_xlim(times[0], times[-1])
    
    plt.suptitle('Group Average: Response vs Feedback Locking', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # --- Print interpretation ---
    print('\n=== Interpretation Guide ===')
    print(f'Mean response-to-feedback delay: {mean_delay:.2f}s')
    print('')
    
    # Find peaks
    resp_peak_idx = np.argmax(mean_resp)
    fb_peak_idx = np.argmax(mean_fb)
    
    print(f'Response-locked peak: {mean_resp[resp_peak_idx]:.0f}% at t={times[resp_peak_idx]:.2f}s')
    print(f'Feedback-locked peak: {mean_fb[fb_peak_idx]:.0f}% at t={times[fb_peak_idx]:.2f}s')
    print('')
    
    # Interpretation
    if times[resp_peak_idx] < 0.5 and times[resp_peak_idx] > -0.5:
        print('→ Response-locked peak is near t=0: Theta is likely RESPONSE-LOCKED')
        print('  (motor preparation, decision-making, or action monitoring)')
    elif times[resp_peak_idx] > mean_delay - 0.5:
        print('→ Response-locked peak is near feedback time: Theta may be FEEDBACK-LOCKED')
    else:
        print('→ Response-locked peak is in the wait period: Theta may be ANTICIPATORY')
    
    return fig


# Plot group average
if len(response_results) > 0:
    fig = plot_group_response_vs_feedback(response_results, channel_idx=0)

In [ ]:
# === CELL: Extract Individual Theta Metrics ===
# Paste this after the feedback-locked analysis (Section 10 or 11)

def extract_theta_metrics(results):
    """
    Extract simple, robust theta metrics for each subject.
    
    These metrics capture individual differences in theta responsiveness
    without requiring precise event timing.
    """
    rows = []
    
    for r in results:
        times = r['times']
        
        # Define windows
        pre_mask = (times >= -0.5) & (times < -0.1)   # Baseline window
        post_mask = (times >= 0.0) & (times < 0.5)    # Post-event (0-500ms)
        late_mask = (times >= 0.5) & (times < 1.0)    # Late window (500-1000ms)
        full_mask = (times >= -0.5) & (times < 1.0)   # Full peri-event
        
        for ch_idx, ch_label in enumerate(EEG_CH_LABELS):
            ersp = r['ersp_all'][:, ch_idx]
            
            # Core metrics
            row = {
                'subject_id': r['subject_id'],
                'run': r['run_num'],
                'channel': ch_label,
                
                # Metric 1: Mean post-event theta (% change from baseline)
                'theta_post_mean': ersp[post_mask].mean(),
                
                # Metric 2: Peak post-event theta
                'theta_post_peak': ersp[post_mask].max(),
                
                # Metric 3: Late theta (500-1000ms) - more likely feedback-related
                'theta_late_mean': ersp[late_mask].mean(),
                
                # Metric 4: Overall theta reactivity (full epoch mean)
                'theta_reactivity': ersp[full_mask].mean(),
                
                # Metric 5: Theta "burst" magnitude (peak - trough in post window)
                'theta_burst_range': ersp[post_mask].max() - ersp[post_mask].min(),
            }
            
            # Add reward vs no-reward difference if available
            if r['ersp_reward'] is not None and r['ersp_no_reward'] is not None:
                rew = r['ersp_reward'][post_mask, ch_idx].mean()
                no_rew = r['ersp_no_reward'][post_mask, ch_idx].mean()
                row['theta_reward_diff'] = no_rew - rew  # Typically no-reward > reward
            else:
                row['theta_reward_diff'] = np.nan
            
            rows.append(row)
    
    return pd.DataFrame(rows)


# Extract metrics from feedback-locked results
if len(all_results) > 0:
    theta_metrics = extract_theta_metrics(all_results)
    
    # Average across runs within subject (for subjects with multiple baseline runs)
    subject_metrics = theta_metrics.groupby(['subject_id', 'channel']).agg({
        'theta_post_mean': 'mean',
        'theta_post_peak': 'mean', 
        'theta_late_mean': 'mean',
        'theta_reactivity': 'mean',
        'theta_burst_range': 'mean',
        'theta_reward_diff': 'mean',
    }).reset_index()
    
    print("=== Per-Subject Theta Metrics (averaged across baseline runs) ===")
    print(f"\nF4 channel metrics:")
    display(subject_metrics[subject_metrics['channel'] == 'F4'].round(1))
    
    # Save
    output_path = OUTPUT_DIR / 'individual_theta_metrics.csv'
    subject_metrics.to_csv(output_path, index=False)
    print(f"\nSaved to: {output_path}")
else:
    print("No results available")

In [ ]:
# === CELL: Visualize Individual Differences ===

def plot_individual_theta(subject_metrics, metric='theta_reactivity'):
    """
    Bar plot showing individual differences in theta metric.
    """
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    for idx, ch in enumerate(EEG_CH_LABELS):
        ax = axes[idx]
        ch_data = subject_metrics[subject_metrics['channel'] == ch].sort_values(metric)
        
        colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(ch_data)))
        ax.barh(ch_data['subject_id'].astype(str), ch_data[metric], color=colors)
        ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel(f'{metric} (% change)')
        ax.set_title(f'{ch}')
        
        if idx == 0:
            ax.set_ylabel('Subject')
    
    plt.suptitle(f'Individual Differences in Theta: {metric}', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Print rank order
    f4_data = subject_metrics[subject_metrics['channel'] == 'F4'].sort_values(metric, ascending=False)
    print(f"\nF4 Rank Order ({metric}):")
    for i, (_, row) in enumerate(f4_data.iterrows(), 1):
        print(f"  {i}. Sub-{row['subject_id']}: {row[metric]:.1f}%")


# Plot the most interpretable metric
plot_individual_theta(subject_metrics, metric='theta_reactivity')
plot_individual_theta(subject_metrics, metric='theta_late_mean')

In [ ]:
# === CELL: Correlate Theta Metrics with Individual Differences ===

def correlate_theta_with_demographics(subject_metrics, demographics_df):
    """
    Correlate theta metrics with age, reward sensitivity, etc.
    
    Parameters:
        subject_metrics: DataFrame from extract_theta_metrics (averaged across runs)
        demographics_df: DataFrame with subject_id, age, SPSRQ_reward, etc.
    """
    from scipy import stats
    
    # Merge
    f4_metrics = subject_metrics[subject_metrics['channel'] == 'F4'].copy()
    f4_metrics['subject_id'] = f4_metrics['subject_id'].astype(str)
    
    # If you have demographics loaded from behavioral data or REDCap:
    # merged = f4_metrics.merge(demographics_df, on='subject_id', how='inner')
    
    # For now, let's use age from the behavioral data
    # We'll need to load it - here's a simple version using what we have:
    
    print("=== Correlations with Individual Differences (F4) ===\n")
    
    theta_vars = ['theta_post_mean', 'theta_late_mean', 'theta_reactivity', 
                  'theta_reward_diff']
    
    return f4_metrics


# First, let's get age from the behavioral data
def get_subject_demographics():
    """Pull age and other demographics from behavioral CSVs."""
    demo_rows = []
    
    for subject_id in SUBJECT_INFO.keys():
        behav_dir = BEHAV_DIR / f'sub-{subject_id}'
        behav_path = get_latest_run_file(behav_dir, 1)  # Use run 1
        
        if behav_path is None:
            continue
            
        df = pd.read_csv(behav_path)
        
        if len(df) > 0:
            row = {
                'subject_id': subject_id,
                'age': df['age'].iloc[0] if 'age' in df.columns else np.nan,
                'gender': df['gender'].iloc[0] if 'gender' in df.columns else np.nan,
            }
            demo_rows.append(row)
    
    return pd.DataFrame(demo_rows)


# Get demographics
demographics = get_subject_demographics()
print("Demographics:")
display(demographics)

# Merge with theta metrics
f4_metrics = subject_metrics[subject_metrics['channel'] == 'F4'].copy()
f4_metrics['subject_id'] = f4_metrics['subject_id'].astype(str)
merged = f4_metrics.merge(demographics, on='subject_id', how='inner')

print("\n=== Correlations with Age ===")
from scipy import stats

theta_vars = ['theta_post_mean', 'theta_late_mean', 'theta_reactivity', 'theta_reward_diff']

for var in theta_vars:
    valid = ~(np.isnan(merged[var]) | np.isnan(merged['age']))
    if valid.sum() >= 3:
        r, p = stats.pearsonr(merged.loc[valid, 'age'], merged.loc[valid, var])
        sig = '*' if p < 0.05 else ''
        print(f"  Age × {var:20s}: r = {r:+.3f}, p = {p:.3f} {sig}")

In [ ]:
# === CELL: Scatter Plots of Theta vs Age ===

def plot_theta_vs_age(merged):
    """Scatter plots of theta metrics vs age."""
    
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes = axes.flatten()
    
    theta_vars = ['theta_post_mean', 'theta_late_mean', 'theta_reactivity', 'theta_reward_diff']
    
    for idx, var in enumerate(theta_vars):
        ax = axes[idx]
        
        valid = ~(np.isnan(merged[var]) | np.isnan(merged['age']))
        x = merged.loc[valid, 'age']
        y = merged.loc[valid, var]
        
        ax.scatter(x, y, s=80, alpha=0.7, edgecolor='black')
        
        # Add subject labels
        for _, row in merged[valid].iterrows():
            ax.annotate(row['subject_id'], (row['age'], row[var]), 
                       fontsize=8, ha='left', va='bottom')
        
        # Regression line
        if len(x) >= 3:
            z = np.polyfit(x, y, 1)
            p = np.poly1d(z)
            x_line = np.linspace(x.min(), x.max(), 100)
            ax.plot(x_line, p(x_line), 'r--', alpha=0.5)
            
            r, pval = stats.pearsonr(x, y)
            ax.set_title(f'{var}\nr = {r:.2f}, p = {pval:.3f}')
        else:
            ax.set_title(var)
        
        ax.set_xlabel('Age')
        ax.set_ylabel('Theta (% change)')
    
    plt.suptitle('Theta Metrics vs Age', fontsize=14)
    plt.tight_layout()
    plt.show()


plot_theta_vs_age(merged)

In [ ]:
# === CELL: Extract ITF from Task Epochs ===

def extract_itf_from_epochs(all_results, channel_idx=0):
    """
    Extract Individual Theta Frequency from task epochs.
    
    Method: Compute power spectrum of epoched data, find peak in theta range.
    This is more likely to show theta peaks than resting-state data.
    """
    from scipy import signal
    
    itf_rows = []
    
    for r in all_results:
        # We need the raw epochs, not ERSP
        # This requires re-running with epoch storage...
        # For now, let's flag this as a TODO
        pass
    
    print("NOTE: ITF extraction requires raw epoch data.")
    print("The current pipeline only stores ERSP (baseline-corrected power).")
    print("To extract ITF, we'd need to modify analyze_subject_run() to also return raw epochs.")
    
    return None


# For now, let's check if we can get ITF from the v5 EEG notebook results
# Those had specparam fits on baseline runs

print("=== ITF from Resting Analysis (v5 notebook) ===")
print("From your earlier analysis, only sub-10951 had detectable theta peak at ~6.3 Hz")
print("Most subjects showed alpha peaks (8-13 Hz) but no clear theta peaks")
print("")
print("This is actually informative:")
print("- Subjects without resting theta peaks may be 'theta-weak' at baseline")  
print("- tACS at 6 Hz could be providing theta that they don't naturally generate")
print("- The task-related theta we're measuring here shows everyone CAN generate theta")
print("  when engaged in the task — tACS might enhance this task-related theta")

In [ ]:
# === CELL: Visualize High vs Low Theta Reactivity ===

def plot_high_vs_low_theta_example():
    """
    Illustrative figure showing what high vs low theta reactivity looks like.
    """
    # Create synthetic time axis (matching our epoch window)
    times = np.linspace(-1.0, 1.5, 250)
    
    # Create example theta power timecourses
    # Both show the same PATTERN (increase after feedback), but different MAGNITUDE
    
    # Baseline (pre-feedback, centered around 0%)
    baseline = np.zeros_like(times)
    
    # High theta reactivity: large increase post-feedback
    high_theta = np.zeros_like(times)
    # Ramp up starting at t=0, peak around 300ms, sustained
    post_mask = times >= 0
    high_theta[post_mask] = 400 * np.exp(-((times[post_mask] - 0.3)**2) / 0.1) + \
                            150 * (1 - np.exp(-times[post_mask] / 0.2))
    # Add some pre-feedback activity
    pre_mask = (times >= -0.5) & (times < 0)
    high_theta[pre_mask] = 50 + 30 * np.sin(times[pre_mask] * 10)
    
    # Low theta reactivity: small increase post-feedback (same pattern, smaller magnitude)
    low_theta = high_theta * 0.25  # Simply scaled down
    
    # Add some noise for realism
    np.random.seed(42)
    high_theta += np.random.randn(len(times)) * 20
    low_theta += np.random.randn(len(times)) * 10
    
    # Smooth
    from scipy.ndimage import gaussian_filter1d
    high_theta = gaussian_filter1d(high_theta, sigma=3)
    low_theta = gaussian_filter1d(low_theta, sigma=3)
    
    # --- PLOTTING ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Panel 1: High theta reactivity subject
    ax1 = axes[0]
    ax1.fill_between(times, 0, high_theta, where=high_theta > 0, alpha=0.3, color='blue')
    ax1.plot(times, high_theta, 'b-', linewidth=2)
    ax1.axvline(0, color='red', linestyle='--', linewidth=2, label='Feedback')
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax1.axvspan(0, 0.5, color='yellow', alpha=0.1, label='Analysis window')
    ax1.set_xlabel('Time from Feedback (s)')
    ax1.set_ylabel('Theta Power (% change)')
    ax1.set_title('HIGH Theta Reactivity\n(e.g., Sub-11773: 295%)', fontsize=12, fontweight='bold')
    ax1.set_xlim(-1, 1.5)
    ax1.set_ylim(-50, 500)
    ax1.legend(loc='upper right', fontsize=8)
    
    # Panel 2: Low theta reactivity subject
    ax2 = axes[1]
    ax2.fill_between(times, 0, low_theta, where=low_theta > 0, alpha=0.3, color='green')
    ax2.plot(times, low_theta, 'g-', linewidth=2)
    ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='Feedback')
    ax2.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax2.axvspan(0, 0.5, color='yellow', alpha=0.1, label='Analysis window')
    ax2.set_xlabel('Time from Feedback (s)')
    ax2.set_ylabel('Theta Power (% change)')
    ax2.set_title('LOW Theta Reactivity\n(e.g., Sub-10951: 49%)', fontsize=12, fontweight='bold')
    ax2.set_xlim(-1, 1.5)
    ax2.set_ylim(-50, 500)  # Same scale for comparison
    ax2.legend(loc='upper right', fontsize=8)
    
    # Panel 3: Overlay comparison
    ax3 = axes[2]
    ax3.plot(times, high_theta, 'b-', linewidth=2, label='High reactivity')
    ax3.plot(times, low_theta, 'g-', linewidth=2, label='Low reactivity')
    ax3.axvline(0, color='red', linestyle='--', linewidth=2)
    ax3.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax3.fill_between(times, low_theta, high_theta, alpha=0.2, color='purple')
    ax3.annotate('', xy=(0.3, high_theta[175]), xytext=(0.3, low_theta[175]),
                 arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
    ax3.text(0.35, (high_theta[175] + low_theta[175])/2, 'Individual\ndifference', 
             fontsize=10, color='purple', va='center')
    ax3.set_xlabel('Time from Feedback (s)')
    ax3.set_ylabel('Theta Power (% change)')
    ax3.set_title('Comparison\n(Same pattern, different magnitude)', fontsize=12, fontweight='bold')
    ax3.set_xlim(-1, 1.5)
    ax3.set_ylim(-50, 500)
    ax3.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    # --- Explanation ---
    print("=" * 70)
    print("INTERPRETATION")
    print("=" * 70)
    print("""
Both subjects show the SAME PATTERN:
  - Low theta power before feedback (baseline)
  - Increase in theta power after feedback (the "response")
  - Gradual return toward baseline

The DIFFERENCE is in MAGNITUDE:
  - High reactivity: Large increase (e.g., 300-400% above baseline)
  - Low reactivity: Small increase (e.g., 50-100% above baseline)

What this might mean for tACS:
  - HIGH baseline reactivity: Already generating strong theta → tACS may be redundant
  - LOW baseline reactivity: Weak endogenous theta → tACS could "boost" the system
  
This is analogous to:
  - A person with naturally high muscle strength may not benefit as much from a strength aid
  - A person with low baseline strength has more "room to improve"
""")
    
    return fig


# Run it
fig = plot_high_vs_low_theta_example()

In [ ]:
# === CELL: Show YOUR actual data - High vs Low subjects ===

def plot_actual_high_vs_low(all_results, high_sub='11773', low_sub='10951', channel_idx=0):
    """
    Plot actual data from a high and low theta reactivity subject.
    """
    # Find the results for these subjects
    high_result = None
    low_result = None
    
    for r in all_results:
        if r['subject_id'] == high_sub:
            high_result = r
        if r['subject_id'] == low_sub:
            low_result = r
    
    if high_result is None or low_result is None:
        print(f"Could not find both subjects in results")
        print(f"Available: {[r['subject_id'] for r in all_results]}")
        return
    
    ch_label = EEG_CH_LABELS[channel_idx]
    times = high_result['times']
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # High subject
    ax1 = axes[0]
    high_ersp = high_result['ersp_all'][:, channel_idx]
    ax1.fill_between(times, 0, high_ersp, where=high_ersp > 0, alpha=0.3, color='blue')
    ax1.plot(times, high_ersp, 'b-', linewidth=2)
    ax1.axvline(0, color='red', linestyle='--', linewidth=2)
    ax1.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax1.set_xlabel('Time from Feedback (s)')
    ax1.set_ylabel('Theta Power (% change)')
    reactivity_high = high_ersp[(times >= -0.5) & (times < 1.0)].mean()
    ax1.set_title(f'Sub-{high_sub} (HIGH)\nReactivity: {reactivity_high:.0f}%')
    ax1.set_xlim(times[0], times[-1])
    
    # Low subject
    ax2 = axes[1]
    low_ersp = low_result['ersp_all'][:, channel_idx]
    ax2.fill_between(times, 0, low_ersp, where=low_ersp > 0, alpha=0.3, color='green')
    ax2.plot(times, low_ersp, 'g-', linewidth=2)
    ax2.axvline(0, color='red', linestyle='--', linewidth=2)
    ax2.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax2.set_xlabel('Time from Feedback (s)')
    ax2.set_ylabel('Theta Power (% change)')
    reactivity_low = low_ersp[(times >= -0.5) & (times < 1.0)].mean()
    ax2.set_title(f'Sub-{low_sub} (LOW)\nReactivity: {reactivity_low:.0f}%')
    ax2.set_xlim(times[0], times[-1])
    
    # Match y-axes
    ymax = max(ax1.get_ylim()[1], ax2.get_ylim()[1])
    ymin = min(ax1.get_ylim()[0], ax2.get_ylim()[0])
    ax1.set_ylim(ymin, ymax)
    ax2.set_ylim(ymin, ymax)
    
    # Overlay
    ax3 = axes[2]
    ax3.plot(times, high_ersp, 'b-', linewidth=2, label=f'Sub-{high_sub} (High)')
    ax3.plot(times, low_ersp, 'g-', linewidth=2, label=f'Sub-{low_sub} (Low)')
    ax3.axvline(0, color='red', linestyle='--', linewidth=2)
    ax3.axhline(0, color='gray', linestyle='-', alpha=0.5)
    ax3.set_xlabel('Time from Feedback (s)')
    ax3.set_ylabel('Theta Power (% change)')
    ax3.set_title(f'Comparison ({ch_label})')
    ax3.set_xlim(times[0], times[-1])
    ax3.set_ylim(ymin, ymax)
    ax3.legend()
    
    plt.suptitle('Actual Data: High vs Low Theta Reactivity Subjects', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    return fig


# Plot actual data
if len(all_results) > 0:
    fig = plot_actual_high_vs_low(all_results, high_sub='11773', low_sub='10951', channel_idx=0)

In [ ]:
# === CELL: Visualize the Alignment Uncertainty ===

def plot_alignment_uncertainty():
    """
    Visual explanation of timestamp alignment uncertainty.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # === Panel 1: The two stopwatches problem ===
    ax1 = axes[0, 0]
    ax1.set_xlim(0, 100)
    ax1.set_ylim(0, 10)
    
    # EEG timeline
    ax1.arrow(5, 7, 85, 0, head_width=0.3, head_length=2, fc='blue', ec='blue')
    ax1.text(50, 8, 'EEG Recording Timeline', ha='center', fontsize=11, color='blue')
    ax1.text(5, 6.2, '0s\n(EEG starts)', ha='center', fontsize=9)
    ax1.text(90, 6.2, '360s\n(EEG ends)', ha='center', fontsize=9)
    
    # Behavioral timeline (offset unknown)
    ax1.arrow(20, 3, 70, 0, head_width=0.3, head_length=2, fc='green', ec='green')
    ax1.text(55, 4, 'Task Timeline', ha='center', fontsize=11, color='green')
    ax1.text(20, 2.2, 'Trial 1', ha='center', fontsize=9)
    ax1.text(90, 2.2, 'Trial 72', ha='center', fontsize=9)
    
    # Unknown offset
    ax1.annotate('', xy=(20, 5), xytext=(5, 5),
                arrowprops=dict(arrowstyle='<->', color='red', lw=2))
    ax1.text(12.5, 5.5, '???', ha='center', fontsize=14, color='red', fontweight='bold')
    ax1.text(12.5, 4.5, 'Unknown\noffset', ha='center', fontsize=9, color='red')
    
    ax1.set_title('The Problem: Two Unsynchronized Timelines', fontsize=12, fontweight='bold')
    ax1.axis('off')
    
    # === Panel 2: What we assume vs reality ===
    ax2 = axes[0, 1]
    
    times = np.linspace(-2, 3, 500)
    
    # Create a "true" theta response (unknown to us)
    true_response = np.zeros_like(times)
    # True feedback at t=0, theta peaks at t=0.3
    true_response = 200 * np.exp(-((times - 0.3)**2) / 0.08)
    true_response += np.random.randn(len(times)) * 10
    from scipy.ndimage import gaussian_filter1d
    true_response = gaussian_filter1d(true_response, sigma=3)
    
    # Plot with different assumed alignments
    ax2.plot(times, true_response, 'k-', linewidth=2, label='If alignment is PERFECT')
    ax2.plot(times - 1, true_response, 'b--', linewidth=2, alpha=0.7, label='If we are 1s too EARLY')
    ax2.plot(times + 1, true_response, 'r--', linewidth=2, alpha=0.7, label='If we are 1s too LATE')
    
    ax2.axvline(0, color='green', linestyle='-', linewidth=3, label='Where we THINK feedback is')
    ax2.axhline(0, color='gray', linestyle='-', alpha=0.3)
    
    ax2.set_xlabel('Time relative to ASSUMED feedback (s)')
    ax2.set_ylabel('Theta Power')
    ax2.set_title('Same Neural Response, Different Alignments', fontsize=12, fontweight='bold')
    ax2.legend(loc='upper right', fontsize=9)
    ax2.set_xlim(-2, 3)
    
    # === Panel 3: Why individual differences are preserved ===
    ax3 = axes[1, 0]
    
    # Two subjects with different reactivity
    high_response = 300 * np.exp(-((times - 0.3)**2) / 0.08)
    low_response = 80 * np.exp(-((times - 0.3)**2) / 0.08)
    
    # Even with wrong alignment (shift by 1s), the DIFFERENCE is preserved
    shift = 1.0  # We're off by 1 second
    times_shifted = times + shift
    
    ax3.plot(times_shifted, high_response, 'b-', linewidth=2, label='High reactivity subject')
    ax3.plot(times_shifted, low_response, 'g-', linewidth=2, label='Low reactivity subject')
    ax3.axvline(0, color='red', linestyle='--', linewidth=2, label='Assumed feedback time')
    ax3.axhline(0, color='gray', linestyle='-', alpha=0.3)
    
    # Shade the analysis window
    ax3.axvspan(0, 0.5, color='yellow', alpha=0.2, label='Analysis window')
    
    # Show the difference is preserved
    idx_window = (times_shifted >= 0) & (times_shifted < 0.5)
    high_mean = high_response[idx_window].mean()
    low_mean = low_response[idx_window].mean()
    
    ax3.annotate(f'High: {high_mean:.0f}%', xy=(0.25, high_mean), fontsize=10, color='blue')
    ax3.annotate(f'Low: {low_mean:.0f}%', xy=(0.25, low_mean), fontsize=10, color='green')
    
    ax3.set_xlabel('Time relative to ASSUMED feedback (s)')
    ax3.set_ylabel('Theta Power (% change)')
    ax3.set_title('Individual Differences PRESERVED Despite Alignment Error', 
                  fontsize=12, fontweight='bold')
    ax3.legend(loc='upper right', fontsize=9)
    ax3.set_xlim(-2, 3)
    
    # === Panel 4: The confidence summary ===
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    confidence_text = """
    WHAT WE'RE CONFIDENT ABOUT:
    
    ✓ Relative timing between trials (Trial 2 was 5s after Trial 1)
    ✓ Relative differences between subjects (A > B in theta power)
    ✓ The overall pattern of theta activity during the task
    ✓ Individual difference metrics (theta_reactivity, etc.)
    
    
    WHAT WE'RE UNCERTAIN ABOUT:
    
    ✗ Exact absolute timing (is the peak at +300ms or +1300ms post-feedback?)
    ✗ Whether the theta burst is response-locked, feedback-locked, or anticipatory
    ✗ Precise latency of effects
    
    
    IMPLICATIONS FOR YOUR STUDY:
    
    → Individual differences in theta reactivity are VALID
    → Correlations with behavior/age/SPSRQ are VALID  
    → Claims about "feedback-related" theta should be CAUTIOUS
    → Future participants: FIX THE LSL CONNECTION!
    """
    
    ax4.text(0.05, 0.95, confidence_text, transform=ax4.transAxes, fontsize=11,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    ax4.set_title('Summary: What Can We Claim?', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return fig


# Run it
fig = plot_alignment_uncertainty()

In [ ]:
# === CELL: How sensitive is theta_reactivity to alignment shifts? ===

def test_alignment_sensitivity(all_results, channel_idx=0):
    """
    Test: If we shift the analysis window by ±0.5s or ±1.0s, 
    how much do the theta_reactivity values change?
    
    If individual differences are PRESERVED across shifts, 
    then between-subject offset variability doesn't matter much.
    """
    from scipy import stats
    
    shifts = [-1.0, -0.5, 0.0, +0.5, +1.0]  # seconds
    
    # Store reactivity at each shift for each subject
    results_by_shift = {shift: {} for shift in shifts}
    
    for r in all_results:
        times = r['times']
        ersp = r['ersp_all'][:, channel_idx]
        sub = r['subject_id']
        
        for shift in shifts:
            # Shifted window: original was -0.5 to 1.0, now shift it
            win_start = -0.5 + shift
            win_end = 1.0 + shift
            
            mask = (times >= win_start) & (times < win_end)
            
            if mask.sum() > 0:
                reactivity = ersp[mask].mean()
                results_by_shift[shift][sub] = reactivity
    
    # Build a DataFrame
    subjects = list(results_by_shift[0.0].keys())
    df = pd.DataFrame({
        f'shift_{shift:+.1f}s': [results_by_shift[shift].get(s, np.nan) for s in subjects]
        for shift in shifts
    }, index=subjects)
    
    print("=== Theta Reactivity at Different Alignment Shifts ===\n")
    print(df.round(1))
    
    # Compute correlations between shift=0 and other shifts
    print("\n=== Correlation with Original (shift=0) ===")
    print("(High correlation = individual differences preserved)\n")
    
    baseline = df['shift_+0.0s']
    
    for shift in shifts:
        if shift == 0:
            continue
        col = f'shift_{shift:+.1f}s'
        valid = ~(baseline.isna() | df[col].isna())
        if valid.sum() >= 3:
            r, p = stats.pearsonr(baseline[valid], df[col][valid])
            print(f"  Shift {shift:+.1f}s vs 0.0s:  r = {r:.3f}, p = {p:.3f}")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Panel 1: Reactivity values at each shift
    ax1 = axes[0]
    for sub in subjects:
        values = [results_by_shift[shift].get(sub, np.nan) for shift in shifts]
        ax1.plot(shifts, values, 'o-', label=f'Sub-{sub}', alpha=0.7)
    ax1.set_xlabel('Alignment Shift (seconds)')
    ax1.set_ylabel('Theta Reactivity (%)')
    ax1.set_title('Reactivity at Different Alignment Shifts')
    ax1.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax1.axvline(0, color='gray', linestyle='--', alpha=0.5)
    
    # Panel 2: Rank order preservation
    ax2 = axes[1]
    # Rank at each shift
    ranks_df = df.rank(ascending=False)
    for sub in subjects:
        ranks = [ranks_df.loc[sub, f'shift_{shift:+.1f}s'] for shift in shifts]
        ax2.plot(shifts, ranks, 'o-', label=f'Sub-{sub}', alpha=0.7)
    ax2.set_xlabel('Alignment Shift (seconds)')
    ax2.set_ylabel('Rank (1 = highest reactivity)')
    ax2.set_title('Rank Order Across Shifts')
    ax2.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.invert_yaxis()  # Rank 1 at top
    
    plt.tight_layout()
    plt.show()
    
    # Summary
    print("\n" + "=" * 60)
    print("INTERPRETATION")
    print("=" * 60)
    
    # Check if rank order is stable
    rank_at_0 = ranks_df['shift_+0.0s'].sort_values()
    rank_at_neg1 = ranks_df['shift_-1.0s'].sort_values()
    rank_at_pos1 = ranks_df['shift_+1.0s'].sort_values()
    
    # Spearman correlation of ranks
    rho_neg, _ = stats.spearmanr(ranks_df['shift_+0.0s'], ranks_df['shift_-1.0s'])
    rho_pos, _ = stats.spearmanr(ranks_df['shift_+0.0s'], ranks_df['shift_+1.0s'])
    
    print(f"""
    Rank-order correlation (Spearman's rho):
      Shift -1.0s vs 0.0s:  ρ = {rho_neg:.3f}
      Shift +1.0s vs 0.0s:  ρ = {rho_pos:.3f}
    
    If ρ > 0.8: Rank order is STABLE — individual differences preserved
    If ρ < 0.5: Rank order is UNSTABLE — alignment variability is a problem
    """)
    
    return df


# Run the sensitivity test
if len(all_results) > 0:
    shift_df = test_alignment_sensitivity(all_results)

In [ ]:
# === CELL: Visualize Raw Theta Power with Epoch Windows (Fixed Pattern) ===

def plot_full_run_with_epochs(subject_id, run_num, channel_idx=0, 
                               show_n_epochs=10, zoom_window=None):
    """
    Show the full run's theta power timecourse with epoch windows overlaid.
    """
    from scipy.signal import hilbert
    
    # --- Find EEG file (pattern: *_sub-{id}_Run {n}.easy) ---
    eeg_path = None
    for f in EEG_DIR.glob(f'*_sub-{subject_id}_Run {run_num}.easy'):
        eeg_path = f
        break
    
    if eeg_path is None:
        print(f"EEG file not found for sub-{subject_id} Run {run_num}")
        print(f"Searched pattern: *_sub-{subject_id}_Run {run_num}.easy")
        matching = list(EEG_DIR.glob(f'*_sub-{subject_id}_*.easy'))
        if matching:
            print(f"Available files for this subject:")
            for f in matching[:5]:
                print(f"  {f.name}")
        return None
    
    print(f"Loading: {eeg_path.name}")
    eeg_data, eeg_times = load_eeg_run(eeg_path)
    ch_label = EEG_CH_LABELS[channel_idx]
    
    # --- Load behavioral data ---
    behav_dir = BEHAV_DIR / f'sub-{subject_id}'
    behav_path = get_latest_run_file(behav_dir, run_num)
    if behav_path is None:
        print(f"Behavioral file not found in {behav_dir}")
        return None
    
    print(f"Loading: {behav_path.name}")
    behav_df = pd.read_csv(behav_path)
    
    # --- Compute feedback times ---
    FEEDBACK_DURATION = 1.0
    behav_df['feedback_time'] = behav_df['timestamp'] - FEEDBACK_DURATION - (behav_df['iti'] / 1000.0)
    feedback_times_relative = behav_df['feedback_time'] - behav_df['feedback_time'].iloc[0]
    
    estimated_first_feedback = 20.0
    feedback_times_eeg = feedback_times_relative + estimated_first_feedback
    
    # --- Preprocess EEG ---
    raw_signal = eeg_data[:, channel_idx].copy()
    theta_filtered = bandpass_filter(raw_signal, THETA_BAND[0], THETA_BAND[1], FS)
    analytic_signal = hilbert(theta_filtered)
    theta_power = np.abs(analytic_signal) ** 2
    
    window_samples = int(0.1 * FS)
    theta_power_smooth = np.convolve(theta_power, np.ones(window_samples)/window_samples, mode='same')
    theta_power_pct = (theta_power_smooth / theta_power_smooth.mean() - 1) * 100
    
    # --- PLOTTING ---
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), 
                              gridspec_kw={'height_ratios': [1, 2, 2, 2]})
    
    # Panel 1: Event markers
    ax0 = axes[0]
    ax0.set_xlim(0, eeg_times[-1])
    ax0.set_ylim(0, 1)
    for i, ft in enumerate(feedback_times_eeg):
        if ft > 0 and ft < eeg_times[-1]:
            color = 'green' if behav_df.iloc[i]['reward'] == 'yes' else 'red'
            ax0.axvline(ft, color=color, alpha=0.5, linewidth=1)
    ax0.set_ylabel('Events')
    ax0.set_yticks([])
    ax0.set_title(f'Sub-{subject_id} Run {run_num} — Full Timeline\n'
                  f'Green = Reward, Red = No Reward (estimated feedback times)',
                  fontsize=12, fontweight='bold')
    
    # Panel 2: Raw EEG
    ax1 = axes[1]
    downsample = 10
    ax1.plot(eeg_times[::downsample], raw_signal[::downsample], 'k-', linewidth=0.3, alpha=0.7)
    ax1.set_ylabel('Raw EEG (µV)')
    ax1.set_xlim(0, eeg_times[-1])
    ax1.set_title(f'{ch_label} — Raw Signal', fontsize=10)
    
    # Panel 3: Theta power (full run)
    ax2 = axes[2]
    ax2.plot(eeg_times[::downsample], theta_power_pct[::downsample], 'b-', linewidth=0.5, alpha=0.8)
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylabel('Theta Power\n(% change from mean)')
    ax2.set_xlim(0, eeg_times[-1])
    ax2.set_title(f'{ch_label} — Continuous Theta Power (4-8 Hz)', fontsize=10)
    
    epochs_to_show = min(show_n_epochs, len(feedback_times_eeg))
    for i in range(epochs_to_show):
        ft = feedback_times_eeg.iloc[i]
        if ft - EPOCH_PRE > 0 and ft + EPOCH_POST < eeg_times[-1]:
            ax2.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.3, zorder=0)
            ax2.axvline(ft, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax2.text(0.02, 0.95, f'Yellow = epoch windows (first {epochs_to_show})\nRed dashed = feedback',
             transform=ax2.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Panel 4: Zoomed view
    ax3 = axes[3]
    if zoom_window is None:
        zoom_start = max(0, estimated_first_feedback - 10)
        zoom_end = zoom_start + 60
    else:
        zoom_start, zoom_end = zoom_window
    
    zoom_mask = (eeg_times >= zoom_start) & (eeg_times <= zoom_end)
    ax3.plot(eeg_times[zoom_mask], theta_power_pct[zoom_mask], 'b-', linewidth=1, alpha=0.9)
    ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax3.set_ylabel('Theta Power\n(% change)')
    ax3.set_xlabel('Time (seconds)')
    ax3.set_xlim(zoom_start, zoom_end)
    ax3.set_title(f'ZOOMED: {zoom_start:.0f}s to {zoom_end:.0f}s', fontsize=10)
    
    for i, ft in enumerate(feedback_times_eeg):
        if ft - EPOCH_PRE >= zoom_start and ft + EPOCH_POST <= zoom_end:
            ax3.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.3, zorder=0)
            ax3.axvline(ft, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
            ax3.text(ft, ax3.get_ylim()[1] * 0.9, f'T{i+1}', ha='center', fontsize=8, color='red')
            rt = ft - 2.3
            if rt >= zoom_start:
                ax3.axvline(rt, color='blue', linestyle=':', linewidth=1, alpha=0.6)
    
    ax3.text(0.02, 0.95, 'Yellow = epoch\nRed -- = feedback\nBlue : = response',
             transform=ax3.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n{'='*60}")
    print(f"SUMMARY: Sub-{subject_id} Run {run_num}")
    print(f"{'='*60}")
    print(f"EEG duration: {eeg_times[-1]:.1f} seconds")
    print(f"Trials: {len(behav_df)}")
    print(f"Feedback range: {feedback_times_eeg.iloc[0]:.1f}s to {feedback_times_eeg.iloc[-1]:.1f}s")
    
    return fig


# --- Run for two subjects ---
print("=" * 70)
print("HIGH THETA REACTIVITY SUBJECT")
print("=" * 70)
fig1 = plot_full_run_with_epochs('11773', 1, channel_idx=0)

print("\n" * 2)
print("=" * 70)
print("LOW THETA REACTIVITY SUBJECT")  
print("=" * 70)
fig2 = plot_full_run_with_epochs('10951', 1, channel_idx=0)

In [ ]:
# === CELL: Check load_eeg_run signature ===
import inspect
print(inspect.signature(load_eeg_run))

# Also try calling it to see what it returns
eeg_path = list(EEG_DIR.glob('*_sub-11773_Run 1.easy'))[0]
result = load_eeg_run(eeg_path)
print(f"\nReturns {len(result)} values:")
print(f"Types: {[type(r).__name__ for r in result]}")

In [ ]:
# === CELL: Check how EEG loading works ===

# Let's see what load_eeg_run returns
eeg_path = list(EEG_DIR.glob('*_sub-11773_Run 1.easy'))[0]
result = load_eeg_run(eeg_path)
print("load_eeg_run returns:")
for i, r in enumerate(result):
    print(f"  [{i}]: {r[:80]}..." if len(r) > 80 else f"  [{i}]: {r}")

# Check if there's a different function for loading raw data
print("\n\nLooking for other EEG loading functions...")
import sys
# List functions that might load EEG
for name in dir():
    if 'eeg' in name.lower() or 'load' in name.lower() or 'easy' in name.lower():
        print(f"  {name}")

In [ ]:
# === CELL: Check the actual return structure ===

eeg_path = list(EEG_DIR.glob('*_sub-11773_Run 1.easy'))[0]
result = load_eeg_run(eeg_path)

# Check if it's a dict-like object
print(f"Type: {type(result)}")

# If it's a dict:
if hasattr(result, 'keys'):
    print(f"Keys: {result.keys()}")
    print(f"eeg shape: {result['eeg'].shape}")
    print(f"time_sec shape: {result['time_sec'].shape}")
    
# If it's a named tuple or similar:
elif hasattr(result, '_fields'):
    print(f"Fields: {result._fields}")
    
# Or access by attribute
elif hasattr(result, 'eeg'):
    print(f"Has .eeg attribute")
    print(f"eeg shape: {result.eeg.shape}")

In [ ]:
# === CELL: Visualize Raw Theta Power with Epoch Windows (Fixed) ===

def plot_full_run_with_epochs(subject_id, run_num, channel_idx=0, 
                               show_n_epochs=10, zoom_window=None):
    """
    Show the full run's theta power timecourse with epoch windows overlaid.
    """
    from scipy.signal import hilbert
    
    # --- Find EEG file ---
    eeg_path = None
    for f in EEG_DIR.glob(f'*_sub-{subject_id}_Run {run_num}.easy'):
        eeg_path = f
        break
    
    if eeg_path is None:
        print(f"EEG file not found for sub-{subject_id} Run {run_num}")
        return None
    
    print(f"Loading: {eeg_path.name}")
    eeg_result = load_eeg_run(eeg_path)
    eeg_data = eeg_result['eeg']
    eeg_times = eeg_result['time_sec']
    ch_label = EEG_CH_LABELS[channel_idx]
    
    # --- Load behavioral data ---
    behav_dir = BEHAV_DIR / f'sub-{subject_id}'
    behav_path = get_latest_run_file(behav_dir, run_num)
    if behav_path is None:
        print(f"Behavioral file not found in {behav_dir}")
        return None
    
    print(f"Loading: {behav_path.name}")
    behav_df = pd.read_csv(behav_path)
    
    # --- Compute feedback times ---
    FEEDBACK_DURATION = 1.0
    behav_df['feedback_time'] = behav_df['timestamp'] - FEEDBACK_DURATION - (behav_df['iti'] / 1000.0)
    feedback_times_relative = behav_df['feedback_time'] - behav_df['feedback_time'].iloc[0]
    
    estimated_first_feedback = 20.0
    feedback_times_eeg = feedback_times_relative + estimated_first_feedback
    
    # --- Preprocess EEG ---
    raw_signal = eeg_data[:, channel_idx].copy()
    theta_filtered = bandpass_filter(raw_signal, THETA_BAND[0], THETA_BAND[1], FS)
    analytic_signal = hilbert(theta_filtered)
    theta_power = np.abs(analytic_signal) ** 2
    
    window_samples = int(0.1 * FS)
    theta_power_smooth = np.convolve(theta_power, np.ones(window_samples)/window_samples, mode='same')
    theta_power_pct = (theta_power_smooth / theta_power_smooth.mean() - 1) * 100
    
    # --- PLOTTING ---
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), 
                              gridspec_kw={'height_ratios': [1, 2, 2, 2]})
    
    # Panel 1: Event markers
    ax0 = axes[0]
    ax0.set_xlim(0, eeg_times[-1])
    ax0.set_ylim(0, 1)
    for i, ft in enumerate(feedback_times_eeg):
        if ft > 0 and ft < eeg_times[-1]:
            color = 'green' if behav_df.iloc[i]['reward'] == 'yes' else 'red'
            ax0.axvline(ft, color=color, alpha=0.5, linewidth=1)
    ax0.set_ylabel('Events')
    ax0.set_yticks([])
    ax0.set_title(f'Sub-{subject_id} Run {run_num} — Full Timeline\n'
                  f'Green = Reward, Red = No Reward (estimated feedback times)',
                  fontsize=12, fontweight='bold')
    
    # Panel 2: Raw EEG
    ax1 = axes[1]
    downsample = 10
    ax1.plot(eeg_times[::downsample], raw_signal[::downsample], 'k-', linewidth=0.3, alpha=0.7)
    ax1.set_ylabel('Raw EEG (µV)')
    ax1.set_xlim(0, eeg_times[-1])
    ax1.set_title(f'{ch_label} — Raw Signal', fontsize=10)
    
    # Panel 3: Theta power (full run)
    ax2 = axes[2]
    ax2.plot(eeg_times[::downsample], theta_power_pct[::downsample], 'b-', linewidth=0.5, alpha=0.8)
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylabel('Theta Power\n(% change from mean)')
    ax2.set_xlim(0, eeg_times[-1])
    ax2.set_title(f'{ch_label} — Continuous Theta Power (4-8 Hz)', fontsize=10)
    
    epochs_to_show = min(show_n_epochs, len(feedback_times_eeg))
    for i in range(epochs_to_show):
        ft = feedback_times_eeg.iloc[i]
        if ft - EPOCH_PRE > 0 and ft + EPOCH_POST < eeg_times[-1]:
            ax2.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.3, zorder=0)
            ax2.axvline(ft, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax2.text(0.02, 0.95, f'Yellow = epoch windows (first {epochs_to_show})\nRed dashed = feedback',
             transform=ax2.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Panel 4: Zoomed view
    ax3 = axes[3]
    if zoom_window is None:
        zoom_start = max(0, estimated_first_feedback - 10)
        zoom_end = zoom_start + 60
    else:
        zoom_start, zoom_end = zoom_window
    
    zoom_mask = (eeg_times >= zoom_start) & (eeg_times <= zoom_end)
    ax3.plot(eeg_times[zoom_mask], theta_power_pct[zoom_mask], 'b-', linewidth=1, alpha=0.9)
    ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax3.set_ylabel('Theta Power\n(% change)')
    ax3.set_xlabel('Time (seconds)')
    ax3.set_xlim(zoom_start, zoom_end)
    ax3.set_title(f'ZOOMED: {zoom_start:.0f}s to {zoom_end:.0f}s', fontsize=10)
    
    for i, ft in enumerate(feedback_times_eeg):
        if ft - EPOCH_PRE >= zoom_start and ft + EPOCH_POST <= zoom_end:
            ax3.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.3, zorder=0)
            ax3.axvline(ft, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
            ax3.text(ft, ax3.get_ylim()[1] * 0.9, f'T{i+1}', ha='center', fontsize=8, color='red')
            rt = ft - 2.3
            if rt >= zoom_start:
                ax3.axvline(rt, color='blue', linestyle=':', linewidth=1, alpha=0.6)
    
    ax3.text(0.02, 0.95, 'Yellow = epoch\nRed -- = feedback\nBlue : = response',
             transform=ax3.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n{'='*60}")
    print(f"SUMMARY: Sub-{subject_id} Run {run_num}")
    print(f"{'='*60}")
    print(f"EEG duration: {eeg_times[-1]:.1f} seconds")
    print(f"Trials: {len(behav_df)}")
    print(f"Feedback range: {feedback_times_eeg.iloc[0]:.1f}s to {feedback_times_eeg.iloc[-1]:.1f}s")
    
    return fig


# --- Run for two subjects ---
print("=" * 70)
print("HIGH THETA REACTIVITY SUBJECT")
print("=" * 70)
fig1 = plot_full_run_with_epochs('11773', 1, channel_idx=0)

print("\n" * 2)
print("=" * 70)
print("LOW THETA REACTIVITY SUBJECT")  
print("=" * 70)
fig2 = plot_full_run_with_epochs('10951', 1, channel_idx=0)

In [ ]:
# === CELL: Check how data is loaded in existing functions ===

# Let's look at how analyze_subject_run or similar functions load data
import inspect

# Check the existing working function
print("=== analyze_subject_run signature and first 50 lines ===")
try:
    source = inspect.getsource(analyze_subject_run)
    print(source[:2000])
except:
    print("Function not found")

print("\n\n=== Or check load_behavioral_run ===")
try:
    source = inspect.getsource(load_behavioral_run)
    print(source[:1000])
except:
    print("Function not found")

In [ ]:
# === CELL: Visualize Raw Theta Power with Epoch Windows (Fixed bandpass call) ===

def plot_full_run_with_epochs(subject_id, run_num, channel_idx=0, 
                               show_n_epochs=10, zoom_window=None):
    """
    Show the full run's theta power timecourse with epoch windows overlaid.
    """
    from scipy.signal import hilbert
    
    # --- Load EEG ---
    eeg_path = find_eeg_run(EEG_DIR, subject_id, run_num)
    if eeg_path is None:
        print(f"EEG file not found for sub-{subject_id} Run {run_num}")
        return None
    
    eeg_data = load_eeg_run(eeg_path)
    print(f"EEG loaded: {eeg_data['duration_sec']:.1f}s")
    
    eeg_times = eeg_data['time_sec']
    ch_label = EEG_CH_LABELS[channel_idx]
    
    # --- Load behavioral ---
    behav_dir = BEHAV_DIR / f'sub-{subject_id}'
    behav_path = get_latest_run_file(behav_dir, run_num)
    if behav_path is None:
        print(f"Behavioral file not found")
        return None
    
    behav_df = load_behavioral_run(behav_path)
    print(f"Behavioral loaded: {len(behav_df)} trials")
    
    # --- Preprocess ---
    sub_info = SUBJECT_INFO.get(subject_id, {})
    has_earclip = sub_info.get('earclip', False)
    eeg_data = preprocess_eeg(eeg_data, has_earclip=has_earclip)
    
    # --- Align timestamps ---
    behav_df = align_timestamps(eeg_data, behav_df)
    
    if 'feedback_time_eeg' in behav_df.columns:
        feedback_times_eeg = behav_df['feedback_time_eeg']
    else:
        print("Available columns:", behav_df.columns.tolist())
        return None
    
    # --- Compute theta power (fixed: pass THETA_BAND as tuple) ---
    raw_signal = eeg_data['eeg'][:, channel_idx].copy()
    theta_filtered = bandpass_filter(raw_signal, THETA_BAND, FS)  # FIXED
    analytic_signal = hilbert(theta_filtered)
    theta_power = np.abs(analytic_signal) ** 2
    
    window_samples = int(0.1 * FS)
    theta_power_smooth = np.convolve(theta_power, np.ones(window_samples)/window_samples, mode='same')
    theta_power_pct = (theta_power_smooth / theta_power_smooth.mean() - 1) * 100
    
    # --- PLOTTING ---
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), 
                              gridspec_kw={'height_ratios': [1, 2, 2, 2]})
    
    # Panel 1: Event markers
    ax0 = axes[0]
    ax0.set_xlim(0, eeg_times[-1])
    ax0.set_ylim(0, 1)
    for i, ft in enumerate(feedback_times_eeg):
        if ft > 0 and ft < eeg_times[-1]:
            color = 'green' if behav_df.iloc[i]['reward'] == 'yes' else 'red'
            ax0.axvline(ft, color=color, alpha=0.5, linewidth=1)
    ax0.set_ylabel('Events')
    ax0.set_yticks([])
    ax0.set_title(f'Sub-{subject_id} Run {run_num} — Full Timeline\n'
                  f'Green = Reward, Red = No Reward (estimated feedback times)',
                  fontsize=12, fontweight='bold')
    
    # Panel 2: Raw EEG
    ax1 = axes[1]
    downsample = 10
    ax1.plot(eeg_times[::downsample], raw_signal[::downsample], 'k-', linewidth=0.3, alpha=0.7)
    ax1.set_ylabel('Raw EEG (µV)')
    ax1.set_xlim(0, eeg_times[-1])
    ax1.set_title(f'{ch_label} — Raw Signal', fontsize=10)
    
    # Panel 3: Theta power (full run)
    ax2 = axes[2]
    ax2.plot(eeg_times[::downsample], theta_power_pct[::downsample], 'b-', linewidth=0.5, alpha=0.8)
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylabel('Theta Power\n(% change from mean)')
    ax2.set_xlim(0, eeg_times[-1])
    ax2.set_title(f'{ch_label} — Continuous Theta Power (4-8 Hz)', fontsize=10)
    
    epochs_to_show = min(show_n_epochs, len(feedback_times_eeg))
    for i in range(epochs_to_show):
        ft = feedback_times_eeg.iloc[i]
        if ft - EPOCH_PRE > 0 and ft + EPOCH_POST < eeg_times[-1]:
            ax2.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.3, zorder=0)
            ax2.axvline(ft, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax2.text(0.02, 0.95, f'Yellow = epoch windows (first {epochs_to_show})\nRed dashed = feedback',
             transform=ax2.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Panel 4: Zoomed view
    ax3 = axes[3]
    first_feedback = feedback_times_eeg.iloc[0]
    if zoom_window is None:
        zoom_start = max(0, first_feedback - 5)
        zoom_end = zoom_start + 60
    else:
        zoom_start, zoom_end = zoom_window
    
    zoom_mask = (eeg_times >= zoom_start) & (eeg_times <= zoom_end)
    ax3.plot(eeg_times[zoom_mask], theta_power_pct[zoom_mask], 'b-', linewidth=1, alpha=0.9)
    ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax3.set_ylabel('Theta Power\n(% change)')
    ax3.set_xlabel('Time (seconds)')
    ax3.set_xlim(zoom_start, zoom_end)
    ax3.set_title(f'ZOOMED: {zoom_start:.0f}s to {zoom_end:.0f}s', fontsize=10)
    
    for i, ft in enumerate(feedback_times_eeg):
        if ft - EPOCH_PRE >= zoom_start and ft + EPOCH_POST <= zoom_end:
            ax3.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.3, zorder=0)
            ax3.axvline(ft, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
            ax3.text(ft, ax3.get_ylim()[1] * 0.9, f'T{i+1}', ha='center', fontsize=8, color='red')
            rt = ft - 2.3
            if rt >= zoom_start:
                ax3.axvline(rt, color='blue', linestyle=':', linewidth=1, alpha=0.6)
    
    ax3.text(0.02, 0.95, 'Yellow = epoch\nRed -- = feedback\nBlue : = response',
             transform=ax3.transAxes, fontsize=9, va='top',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n{'='*60}")
    print(f"SUMMARY: Sub-{subject_id} Run {run_num}")
    print(f"{'='*60}")
    print(f"EEG duration: {eeg_times[-1]:.1f} seconds")
    print(f"Trials: {len(behav_df)}")
    print(f"Feedback range: {feedback_times_eeg.iloc[0]:.1f}s to {feedback_times_eeg.iloc[-1]:.1f}s")
    
    return fig


# --- Run for two subjects ---
print("=" * 70)
print("HIGH THETA REACTIVITY SUBJECT")
print("=" * 70)
fig1 = plot_full_run_with_epochs('11773', 1, channel_idx=0)

print("\n" * 2)
print("=" * 70)
print("LOW THETA REACTIVITY SUBJECT")  
print("=" * 70)
fig2 = plot_full_run_with_epochs('10951', 1, channel_idx=0)

In [ ]:
# === CELL: Plot with artifact rejection ===

def plot_full_run_with_epochs_v2(subject_id, run_num, channel_idx=0, 
                                  show_n_epochs=10, zoom_window=None,
                                  artifact_thresh_uv=150, skip_first_sec=5):
    """
    Same as before, but with artifact masking and option to skip startup.
    """
    from scipy.signal import hilbert
    
    # --- Load data (same as before) ---
    eeg_path = find_eeg_run(EEG_DIR, subject_id, run_num)
    if eeg_path is None:
        return None
    
    eeg_data = load_eeg_run(eeg_path)
    eeg_times = eeg_data['time_sec']
    
    behav_dir = BEHAV_DIR / f'sub-{subject_id}'
    behav_path = get_latest_run_file(behav_dir, run_num)
    if behav_path is None:
        return None
    
    behav_df = load_behavioral_run(behav_path)
    
    sub_info = SUBJECT_INFO.get(subject_id, {})
    has_earclip = sub_info.get('earclip', False)
    eeg_data = preprocess_eeg(eeg_data, has_earclip=has_earclip)
    behav_df = align_timestamps(eeg_data, behav_df)
    feedback_times_eeg = behav_df['feedback_time_eeg']
    
    # --- Compute theta power ---
    raw_signal = eeg_data['eeg'][:, channel_idx].copy()
    theta_filtered = bandpass_filter(raw_signal, THETA_BAND, FS)
    analytic_signal = hilbert(theta_filtered)
    theta_power = np.abs(analytic_signal) ** 2
    
    # --- ARTIFACT MASKING ---
    artifact_mask = np.abs(raw_signal) > artifact_thresh_uv
    
    # Also mask first N seconds (startup artifacts)
    startup_mask = eeg_times < skip_first_sec
    
    combined_mask = artifact_mask | startup_mask
    artifact_pct = 100 * combined_mask.sum() / len(combined_mask)
    
    # Set artifact periods to NaN
    theta_power_clean = theta_power.copy()
    theta_power_clean[combined_mask] = np.nan
    
    # Smooth (ignoring NaN)
    window_samples = int(0.1 * FS)
    theta_power_smooth = np.convolve(theta_power, np.ones(window_samples)/window_samples, mode='same')
    theta_power_smooth_clean = theta_power_smooth.copy()
    theta_power_smooth_clean[combined_mask] = np.nan
    
    # Percent change from CLEAN mean (excluding artifacts)
    clean_mean = np.nanmean(theta_power_smooth_clean)
    theta_power_pct = (theta_power_smooth / clean_mean - 1) * 100
    theta_power_pct_clean = theta_power_pct.copy()
    theta_power_pct_clean[combined_mask] = np.nan
    
    print(f"Sub-{subject_id} Run {run_num}: {artifact_pct:.1f}% of data masked")
    
    # --- PLOTTING ---
    fig, axes = plt.subplots(4, 1, figsize=(16, 12), 
                              gridspec_kw={'height_ratios': [1, 2, 2, 2]})
    
    # Panel 1: Event markers
    ax0 = axes[0]
    ax0.set_xlim(0, eeg_times[-1])
    ax0.set_ylim(0, 1)
    for i, ft in enumerate(feedback_times_eeg):
        if ft > 0 and ft < eeg_times[-1]:
            color = 'green' if behav_df.iloc[i]['reward'] == 'yes' else 'red'
            ax0.axvline(ft, color=color, alpha=0.5, linewidth=1)
    ax0.axvspan(0, skip_first_sec, color='gray', alpha=0.3)  # Show skipped region
    ax0.set_ylabel('Events')
    ax0.set_yticks([])
    ax0.set_title(f'Sub-{subject_id} Run {run_num}\n'
                  f'Gray = excluded startup, {artifact_pct:.1f}% total masked',
                  fontsize=12, fontweight='bold')
    
    # Panel 2: Raw EEG with artifact threshold
    ax1 = axes[1]
    downsample = 10
    ax1.plot(eeg_times[::downsample], raw_signal[::downsample], 'k-', linewidth=0.3, alpha=0.7)
    ax1.axhline(artifact_thresh_uv, color='red', linestyle='--', alpha=0.5)
    ax1.axhline(-artifact_thresh_uv, color='red', linestyle='--', alpha=0.5)
    ax1.axvspan(0, skip_first_sec, color='gray', alpha=0.3)
    ax1.set_ylabel('Raw EEG (µV)')
    ax1.set_xlim(0, eeg_times[-1])
    ax1.set_title(f'F4 — Raw Signal (red = ±{artifact_thresh_uv}µV threshold)', fontsize=10)
    
    # Panel 3: Theta power (CLEAN, capped y-axis)
    ax2 = axes[2]
    ax2.plot(eeg_times[::downsample], theta_power_pct_clean[::downsample], 'b-', linewidth=0.5, alpha=0.8)
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.axvspan(0, skip_first_sec, color='gray', alpha=0.3)
    ax2.set_ylabel('Theta Power\n(% change, artifacts masked)')
    ax2.set_xlim(0, eeg_times[-1])
    
    # Cap y-axis at reasonable range
    clean_data = theta_power_pct_clean[~np.isnan(theta_power_pct_clean)]
    if len(clean_data) > 0:
        ymax = min(np.percentile(clean_data, 99), 500)
        ymin = max(np.percentile(clean_data, 1), -100)
        ax2.set_ylim(ymin, ymax)
    
    ax2.set_title(f'F4 — Continuous Theta Power (artifacts masked, y-axis capped)', fontsize=10)
    
    epochs_to_show = min(show_n_epochs, len(feedback_times_eeg))
    for i in range(epochs_to_show):
        ft = feedback_times_eeg.iloc[i]
        if ft - EPOCH_PRE > skip_first_sec and ft + EPOCH_POST < eeg_times[-1]:
            ax2.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.3, zorder=0)
            ax2.axvline(ft, color='red', linestyle='--', linewidth=1, alpha=0.7)
    
    # Panel 4: Zoomed view (after startup)
    ax3 = axes[3]
    first_valid_feedback = feedback_times_eeg[feedback_times_eeg > skip_first_sec].iloc[0]
    if zoom_window is None:
        zoom_start = max(skip_first_sec, first_valid_feedback - 5)
        zoom_end = zoom_start + 60
    else:
        zoom_start, zoom_end = zoom_window
    
    zoom_mask = (eeg_times >= zoom_start) & (eeg_times <= zoom_end)
    ax3.plot(eeg_times[zoom_mask], theta_power_pct_clean[zoom_mask], 'b-', linewidth=1, alpha=0.9)
    ax3.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax3.set_ylabel('Theta Power\n(% change)')
    ax3.set_xlabel('Time (seconds)')
    ax3.set_xlim(zoom_start, zoom_end)
    ax3.set_title(f'ZOOMED: {zoom_start:.0f}s to {zoom_end:.0f}s (after startup)', fontsize=10)
    
    # Match y-axis to panel 3
    ax3.set_ylim(ax2.get_ylim())
    
    for i, ft in enumerate(feedback_times_eeg):
        if ft - EPOCH_PRE >= zoom_start and ft + EPOCH_POST <= zoom_end:
            ax3.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.3, zorder=0)
            ax3.axvline(ft, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
            ax3.text(ft, ax3.get_ylim()[1] * 0.9, f'T{i+1}', ha='center', fontsize=8, color='red')
    
    plt.tight_layout()
    plt.show()
    
    return fig


# --- Re-run with artifact masking ---
print("=" * 70)
print("SUB-11773 (was 'high reactivity') — WITH ARTIFACT MASKING")
print("=" * 70)
fig1 = plot_full_run_with_epochs_v2('11773', 1, artifact_thresh_uv=150, skip_first_sec=10)

print("\n")
print("=" * 70)
print("SUB-10951 (electrode issues noted) — WITH ARTIFACT MASKING")
print("=" * 70)
fig2 = plot_full_run_with_epochs_v2('10951', 1, artifact_thresh_uv=150, skip_first_sec=10)

In [ ]:
# === CELL: QC Visualization for All Subjects ===

def qc_all_subjects(run_nums=[1, 5], channel_idx=0, artifact_thresh_uv=150, skip_first_sec=10):
    """
    Generate QC plots for all subjects' baseline runs.
    
    Parameters:
        run_nums: Which runs to check (1 and 5 are baseline runs)
        channel_idx: Which channel (0=F4)
        artifact_thresh_uv: Artifact threshold in microvolts
        skip_first_sec: Seconds to skip at start
    """
    from scipy.signal import hilbert
    
    qc_results = []
    
    for subject_id in SUBJECT_INFO.keys():
        for run_num in run_nums:
            
            print(f"\n{'='*70}")
            print(f"Sub-{subject_id} Run {run_num}")
            print('='*70)
            
            # --- Load data ---
            eeg_path = find_eeg_run(EEG_DIR, subject_id, run_num)
            if eeg_path is None:
                print("  EEG file not found — SKIPPING")
                qc_results.append({
                    'subject_id': subject_id,
                    'run': run_num,
                    'status': 'missing',
                    'artifact_pct': np.nan,
                    'mean_theta_clean': np.nan,
                    'max_theta_clean': np.nan,
                    'notes': SUBJECT_INFO[subject_id].get('notes', '')
                })
                continue
            
            eeg_data = load_eeg_run(eeg_path)
            eeg_times = eeg_data['time_sec']
            
            behav_dir = BEHAV_DIR / f'sub-{subject_id}'
            behav_path = get_latest_run_file(behav_dir, run_num)
            if behav_path is None:
                print("  Behavioral file not found — SKIPPING")
                qc_results.append({
                    'subject_id': subject_id,
                    'run': run_num,
                    'status': 'missing_behav',
                    'artifact_pct': np.nan,
                    'mean_theta_clean': np.nan,
                    'max_theta_clean': np.nan,
                    'notes': SUBJECT_INFO[subject_id].get('notes', '')
                })
                continue
            
            behav_df = load_behavioral_run(behav_path)
            
            sub_info = SUBJECT_INFO.get(subject_id, {})
            has_earclip = sub_info.get('earclip', False)
            eeg_data = preprocess_eeg(eeg_data, has_earclip=has_earclip)
            behav_df = align_timestamps(eeg_data, behav_df)
            
            if 'feedback_time_eeg' not in behav_df.columns:
                print("  Alignment failed — SKIPPING")
                continue
            
            feedback_times_eeg = behav_df['feedback_time_eeg']
            
            # --- Compute theta power ---
            raw_signal = eeg_data['eeg'][:, channel_idx].copy()
            theta_filtered = bandpass_filter(raw_signal, THETA_BAND, FS)
            analytic_signal = hilbert(theta_filtered)
            theta_power = np.abs(analytic_signal) ** 2
            
            # --- Artifact masking ---
            artifact_mask = np.abs(raw_signal) > artifact_thresh_uv
            startup_mask = eeg_times < skip_first_sec
            combined_mask = artifact_mask | startup_mask
            artifact_pct = 100 * combined_mask.sum() / len(combined_mask)
            
            # Compute clean theta power
            window_samples = int(0.1 * FS)
            theta_power_smooth = np.convolve(theta_power, np.ones(window_samples)/window_samples, mode='same')
            theta_power_smooth_clean = theta_power_smooth.copy()
            theta_power_smooth_clean[combined_mask] = np.nan
            
            clean_mean = np.nanmean(theta_power_smooth_clean)
            if clean_mean > 0:
                theta_power_pct = (theta_power_smooth / clean_mean - 1) * 100
                theta_power_pct_clean = theta_power_pct.copy()
                theta_power_pct_clean[combined_mask] = np.nan
            else:
                theta_power_pct_clean = np.full_like(theta_power_smooth, np.nan)
            
            # Summary stats
            mean_theta = np.nanmean(theta_power_pct_clean)
            max_theta = np.nanmax(theta_power_pct_clean) if not np.all(np.isnan(theta_power_pct_clean)) else np.nan
            
            # Determine status
            if artifact_pct > 30:
                status = 'BAD - too many artifacts'
            elif artifact_pct > 15:
                status = 'CAUTION - moderate artifacts'
            elif np.isnan(mean_theta) or mean_theta < -50:
                status = 'BAD - no theta signal'
            else:
                status = 'OK'
            
            print(f"  Artifacts: {artifact_pct:.1f}%")
            print(f"  Mean theta (clean): {mean_theta:.1f}%")
            print(f"  Max theta (clean): {max_theta:.1f}%")
            print(f"  Status: {status}")
            if sub_info.get('notes'):
                print(f"  Notes: {sub_info['notes']}")
            
            qc_results.append({
                'subject_id': subject_id,
                'run': run_num,
                'status': status,
                'artifact_pct': artifact_pct,
                'mean_theta_clean': mean_theta,
                'max_theta_clean': max_theta,
                'earclip': has_earclip,
                'notes': sub_info.get('notes', '')
            })
            
            # --- PLOTTING ---
            fig, axes = plt.subplots(3, 1, figsize=(16, 8), 
                                      gridspec_kw={'height_ratios': [1, 2, 2]})
            
            # Panel 1: Events
            ax0 = axes[0]
            ax0.set_xlim(0, eeg_times[-1])
            ax0.set_ylim(0, 1)
            for i, ft in enumerate(feedback_times_eeg):
                if ft > 0 and ft < eeg_times[-1]:
                    color = 'green' if behav_df.iloc[i]['reward'] == 'yes' else 'red'
                    ax0.axvline(ft, color=color, alpha=0.5, linewidth=1)
            ax0.axvspan(0, skip_first_sec, color='gray', alpha=0.3)
            ax0.set_ylabel('Events')
            ax0.set_yticks([])
            
            status_color = 'green' if 'OK' in status else ('orange' if 'CAUTION' in status else 'red')
            ax0.set_title(f'Sub-{subject_id} Run {run_num} | {artifact_pct:.1f}% masked | {status}',
                         fontsize=12, fontweight='bold', color=status_color)
            
            # Panel 2: Raw EEG
            ax1 = axes[1]
            downsample = 10
            ax1.plot(eeg_times[::downsample], raw_signal[::downsample], 'k-', linewidth=0.3, alpha=0.7)
            ax1.axhline(artifact_thresh_uv, color='red', linestyle='--', alpha=0.5)
            ax1.axhline(-artifact_thresh_uv, color='red', linestyle='--', alpha=0.5)
            ax1.axvspan(0, skip_first_sec, color='gray', alpha=0.3)
            ax1.set_ylabel('Raw EEG (µV)')
            ax1.set_xlim(0, eeg_times[-1])
            
            # Auto-scale y-axis but cap at reasonable range
            raw_clean = raw_signal[~combined_mask]
            if len(raw_clean) > 0:
                ymax = min(np.percentile(np.abs(raw_clean), 99.5) * 1.5, 500)
                ax1.set_ylim(-ymax, ymax)
            
            # Panel 3: Theta power
            ax2 = axes[2]
            ax2.plot(eeg_times[::downsample], theta_power_pct_clean[::downsample], 'b-', linewidth=0.5, alpha=0.8)
            ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
            ax2.axvspan(0, skip_first_sec, color='gray', alpha=0.3)
            ax2.set_ylabel('Theta Power\n(% change)')
            ax2.set_xlabel('Time (seconds)')
            ax2.set_xlim(0, eeg_times[-1])
            
            # Cap y-axis
            clean_data = theta_power_pct_clean[~np.isnan(theta_power_pct_clean)]
            if len(clean_data) > 0:
                ymax = min(np.percentile(clean_data, 99), 500)
                ymin = max(np.percentile(clean_data, 1), -100)
                ax2.set_ylim(ymin, ymax)
            
            # Mark epoch windows (first 10)
            for i, ft in enumerate(feedback_times_eeg[:10]):
                if ft - EPOCH_PRE > skip_first_sec and ft + EPOCH_POST < eeg_times[-1]:
                    ax2.axvspan(ft - EPOCH_PRE, ft + EPOCH_POST, color='yellow', alpha=0.2, zorder=0)
            
            plt.tight_layout()
            plt.show()
    
    # --- Summary table ---
    print("\n" + "="*80)
    print("QC SUMMARY")
    print("="*80)
    
    qc_df = pd.DataFrame(qc_results)
    
    # Display with status highlighting
    print("\n")
    display(qc_df[['subject_id', 'run', 'status', 'artifact_pct', 'mean_theta_clean', 'max_theta_clean', 'earclip', 'notes']].round(1))
    
    # Summary counts
    print("\n--- Status Counts ---")
    print(qc_df['status'].value_counts())
    
    return qc_df


# Run QC on all subjects, baseline runs only (1 and 5)
qc_df = qc_all_subjects(run_nums=[1, 5])

In [ ]:
# === CELL: Refined QC Metrics ===

def compute_refined_theta_metrics(qc_df):
    """
    Recompute theta metrics using percentiles instead of max.
    """
    from scipy.signal import hilbert
    
    refined_results = []
    
    for subject_id in SUBJECT_INFO.keys():
        for run_num in [1, 5]:
            
            # Load data
            eeg_path = find_eeg_run(EEG_DIR, subject_id, run_num)
            if eeg_path is None:
                continue
            
            eeg_data = load_eeg_run(eeg_path)
            eeg_times = eeg_data['time_sec']
            
            behav_dir = BEHAV_DIR / f'sub-{subject_id}'
            behav_path = get_latest_run_file(behav_dir, run_num)
            if behav_path is None:
                continue
            
            behav_df = load_behavioral_run(behav_path)
            
            sub_info = SUBJECT_INFO.get(subject_id, {})
            has_earclip = sub_info.get('earclip', False)
            eeg_data = preprocess_eeg(eeg_data, has_earclip=has_earclip)
            
            # Compute theta
            raw_signal = eeg_data['eeg'][:, 0].copy()  # F4
            theta_filtered = bandpass_filter(raw_signal, THETA_BAND, FS)
            analytic_signal = hilbert(theta_filtered)
            theta_power = np.abs(analytic_signal) ** 2
            
            # Artifact mask
            artifact_thresh_uv = 150
            skip_first_sec = 10
            artifact_mask = np.abs(raw_signal) > artifact_thresh_uv
            startup_mask = eeg_times < skip_first_sec
            combined_mask = artifact_mask | startup_mask
            
            # Smooth and normalize
            window_samples = int(0.1 * FS)
            theta_power_smooth = np.convolve(theta_power, np.ones(window_samples)/window_samples, mode='same')
            theta_power_smooth[combined_mask] = np.nan
            
            clean_mean = np.nanmean(theta_power_smooth)
            if clean_mean > 0:
                theta_pct = (theta_power_smooth / clean_mean - 1) * 100
            else:
                continue
            
            # Compute robust metrics
            clean_vals = theta_pct[~np.isnan(theta_pct)]
            if len(clean_vals) == 0:
                continue
            
            refined_results.append({
                'subject_id': subject_id,
                'run': run_num,
                'theta_median': np.median(clean_vals),
                'theta_p75': np.percentile(clean_vals, 75),
                'theta_p95': np.percentile(clean_vals, 95),
                'theta_p99': np.percentile(clean_vals, 99),
                'theta_max': np.max(clean_vals),
                'artifact_pct': 100 * combined_mask.sum() / len(combined_mask),
                'earclip': has_earclip
            })
    
    refined_df = pd.DataFrame(refined_results)
    return refined_df


# Compute refined metrics
refined_df = compute_refined_theta_metrics(qc_df)

print("=== Refined Theta Metrics (F4) ===\n")
print("Using percentiles instead of max to reduce artifact sensitivity:\n")
display(refined_df.round(1))

# Flag potential outliers
print("\n=== Potential Outliers (p95 > 500% or p99 > 1000%) ===")
outliers = refined_df[(refined_df['theta_p95'] > 500) | (refined_df['theta_p99'] > 1000)]
if len(outliers) > 0:
    display(outliers)
else:
    print("None detected")

# Compute subject-level averages (across runs)
print("\n=== Subject-Level Averages (mean of Run 1 + Run 5) ===")
subject_avg = refined_df.groupby('subject_id').agg({
    'theta_p75': 'mean',
    'theta_p95': 'mean',
    'artifact_pct': 'mean',
    'earclip': 'first'
}).round(1).sort_values('theta_p95', ascending=False)

display(subject_avg)

In [ ]:
# === CELL: Final Theta Reactivity Ranking ===

# Exclude problematic runs
clean_runs = refined_df[
    (refined_df['artifact_pct'] < 10) &  # Exclude high-artifact runs
    (refined_df['theta_p95'] > 0) &       # Exclude nonsensical values
    (refined_df['theta_p99'] < 1500)      # Exclude likely artifact-contaminated
].copy()

print("=== Clean Runs Only ===")
display(clean_runs[['subject_id', 'run', 'theta_p95', 'artifact_pct']].round(1))

# Subject-level average from clean runs
clean_subject_avg = clean_runs.groupby('subject_id').agg({
    'theta_p95': 'mean',
    'theta_p75': 'mean',
    'run': 'count'  # How many clean runs
}).rename(columns={'run': 'n_clean_runs'}).round(1)

clean_subject_avg = clean_subject_avg.sort_values('theta_p95', ascending=False)

print("\n=== Final Theta Reactivity Ranking (p95, clean runs only) ===\n")
display(clean_subject_avg)

# Save for later use
clean_subject_avg.to_csv(OUTPUT_DIR / 'theta_reactivity_qc.csv')
print(f"\nSaved to: {OUTPUT_DIR / 'theta_reactivity_qc.csv'}")

In [ ]:
# === CELL: Reliability Analysis - Run 1 vs Run 5 Correlation ===

def compute_run_reliability(refined_df):
    """
    Correlate theta metrics between Run 1 and Run 5 for subjects with both.
    """
    from scipy import stats
    
    # Pivot to get Run 1 and Run 5 as columns
    run1 = refined_df[refined_df['run'] == 1].set_index('subject_id')
    run5 = refined_df[refined_df['run'] == 5].set_index('subject_id')
    
    # Find subjects with both runs (and both meeting QC)
    common_subs = run1.index.intersection(run5.index)
    
    print(f"=== Test-Retest Reliability: Run 1 vs Run 5 ===")
    print(f"Subjects with both baseline runs: {len(common_subs)}")
    print(f"Subjects: {list(common_subs)}\n")
    
    if len(common_subs) < 3:
        print("Too few subjects for reliability analysis")
        return None
    
    # Metrics to check
    metrics = ['theta_p75', 'theta_p95', 'theta_p99']
    
    reliability_results = []
    
    for metric in metrics:
        r1_vals = run1.loc[common_subs, metric].values
        r5_vals = run5.loc[common_subs, metric].values
        
        # Pearson correlation
        r, p = stats.pearsonr(r1_vals, r5_vals)
        
        # ICC (simplified: treating as two raters)
        # ICC(3,1) = (MS_subjects - MS_error) / (MS_subjects + MS_error)
        mean_r1 = np.mean(r1_vals)
        mean_r5 = np.mean(r5_vals)
        grand_mean = (mean_r1 + mean_r5) / 2
        
        # Between-subject variance
        subject_means = (r1_vals + r5_vals) / 2
        ms_subjects = np.var(subject_means, ddof=1) * 2  # *2 for k=2 measurements
        
        # Within-subject variance (error)
        diffs = r1_vals - r5_vals
        ms_error = np.var(diffs, ddof=1) / 2
        
        icc = (ms_subjects - ms_error) / (ms_subjects + ms_error) if (ms_subjects + ms_error) > 0 else np.nan
        
        reliability_results.append({
            'metric': metric,
            'r': r,
            'p': p,
            'ICC': icc,
            'n': len(common_subs)
        })
        
        print(f"{metric}:")
        print(f"  Pearson r = {r:.3f}, p = {p:.3f}")
        print(f"  ICC(3,1) ≈ {icc:.3f}")
        print()
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        r1_vals = run1.loc[common_subs, metric].values
        r5_vals = run5.loc[common_subs, metric].values
        
        ax.scatter(r1_vals, r5_vals, s=80, alpha=0.7, edgecolor='black')
        
        # Add subject labels
        for sub in common_subs:
            ax.annotate(sub, (run1.loc[sub, metric], run5.loc[sub, metric]),
                       fontsize=8, ha='left', va='bottom')
        
        # Identity line
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
                max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'k--', alpha=0.3, label='Identity')
        
        # Regression line
        z = np.polyfit(r1_vals, r5_vals, 1)
        p_line = np.poly1d(z)
        ax.plot(sorted(r1_vals), p_line(sorted(r1_vals)), 'r-', alpha=0.5)
        
        r, p = stats.pearsonr(r1_vals, r5_vals)
        ax.set_xlabel(f'Run 1 {metric}')
        ax.set_ylabel(f'Run 5 {metric}')
        ax.set_title(f'{metric}\nr = {r:.2f}, p = {p:.3f}')
        ax.set_aspect('equal', adjustable='box')
    
    plt.suptitle('Test-Retest Reliability: Baseline Runs', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Interpretation
    print("=" * 60)
    print("INTERPRETATION")
    print("=" * 60)
    print("""
    ICC Benchmarks (Cicchetti, 1994):
      < 0.40  = Poor
      0.40-0.59 = Fair  
      0.60-0.74 = Good
      0.75-1.00 = Excellent
    
    Pearson r Benchmarks:
      < 0.30  = Weak
      0.30-0.50 = Moderate
      0.50-0.70 = Strong
      > 0.70  = Very strong
    """)
    
    return pd.DataFrame(reliability_results)


# Run reliability analysis
# Use the clean_runs dataframe from earlier (excluding artifact-contaminated runs)
reliability_df = compute_run_reliability(clean_runs)

## 13. Notes and Limitations

### Current Limitations

1. **Clock synchronization**: We assume minimal drift between EEG and behavioral computers. For a 6-minute run, this is typically acceptable (<50ms drift), but should be verified if possible.

2. **No LSL markers**: Without hardware event markers, we rely entirely on post-hoc timestamp alignment. Future data collection should use LSL triggers.

3. **Limited channels**: Only 3 EEG channels (F4, P4, P3) are available; stimulation channels are blanked.

4. **Reference**: Subjects without earclip use software average re-reference, which can attenuate signals.

### Future Directions

1. **Time-frequency analysis**: Compute full spectrogram to examine theta, alpha, and other bands.

2. **Phase analysis**: Examine phase-locking to feedback onset.

3. **Trial-level modeling**: Relate theta to behavioral variables (RT, PE, choice).

4. **tACS effects**: Compare baseline vs post-stimulation runs.